# Map 50 — Pathology

Components: pathology foundation and mapping reconciliation. This notebook is executed by `map_pipeline` in the shared map runtime.

In [0]:
if "_PIPELINE_RUN_ID" not in globals():
    raise RuntimeError("Run this component through map_pipeline so shared contracts, checkpoints and audit state are initialized.")

if "_pipeline_resume_skip_component" not in globals():
    def _pipeline_resume_skip_component(component_name: str, target_tables) -> bool:
        complete = _pipeline_resume_component_complete(component_name, target_tables)
        if not complete:
            return False
        print(
            f"[RESUME] {component_name}: durable completion marker found; "
            "skipping completed canonical work"
        )
        _pipeline_audit(
            None,
            "COMPONENT_RESUME_SKIP",
            {"component": component_name, "targets": list(target_tables)},
        )
        return True

In [0]:
_pipeline_component_start("map_pathology")
_pipeline_shared_update_table = globals().get('update_table')
_pipeline_shared_table_exists = globals().get('table_exists')
_pipeline_shared_get_max_timestamp = globals().get('get_max_timestamp')
_pipeline_shared_has_cdf_enabled = globals().get('has_cdf_enabled')
_pipeline_shared_get_incremental = globals().get('get_incremental_data_with_cdf')
globals().pop('_map_updates_original_update_table', None)
globals().pop('_map_medical_personnel_original_update_table', None)
globals().pop('_MAP_UPDATES_ORIGINAL_UPDATE_TABLE', None)

from datetime import datetime, timedelta, timezone
from functools import reduce
import re
import uuid
from delta.tables import DeltaTable
from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F
from pyspark.sql.types import BooleanType, LongType, StringType, StructField, StructType, TimestampType
MP_VERSION = '2.0.0'
TARGET_SCHEMA = '4_prod.bronze'
MAP_SCHEMA = '3_lookup.omop'
MP_TARGET = f'{TARGET_SCHEMA}.map_pathology__canonical'
MP_FLAG = f'{MAP_SCHEMA}.pathology_map_rebuild_flag'
MP_STATE = f'{MAP_SCHEMA}.pathology_pipeline_state_v2'
MP_RUN_LOG = f'{MAP_SCHEMA}.pathology_pipeline_run_log_v2'
MP_BASELINE = f'{MAP_SCHEMA}.pathology_map_baseline_v2'
MP_NATIVE_TEST = f'{MAP_SCHEMA}.pathology_native_test_crosswalk'
MP_NATIVE_RESULT = f'{MAP_SCHEMA}.pathology_native_result_crosswalk'
TEST_MAP = f'{MAP_SCHEMA}.pathology_test_concept_map'
RESULT_MAP = f'{MAP_SCHEMA}.pathology_result_concept_map'
UNIT_MAP = f'{MAP_SCHEMA}.pathology_unit_map'
EXCL_TBL = f'{MAP_SCHEMA}.pathology_result_value_exclusions'
CONCEPT = '3_lookup.omop.concept'
CE = '4_prod.raw.mill_clinical_event'
ORDERS = '4_prod.raw.mill_orders'
PERSON_ALIAS = '4_prod.raw.mill_person_alias'
RESULT_LEVEL = '4_prod.raw.path_patient_resultlevel'
SAMPLE_LEVEL = '4_prod.raw.path_patient_samplelevel'
MASTER_RESULT = '4_prod.raw.path_master_resultable'
ORDER_CATALOG = '3_lookup.mill.mill_order_catalog'
CODE_VALUE = '3_lookup.mill.mill_code_value'
SOURCE_TABLES = {'mill_clinical_event': (CE, 'ADC_UPDT'), 'mill_orders': (ORDERS, 'ADC_UPDT'), 'mill_person_alias': (PERSON_ALIAS, 'ADC_UPDT'), 'path_patient_resultlevel': (RESULT_LEVEL, 'ADC_UPDT'), 'path_patient_samplelevel': (SAMPLE_LEVEL, 'ADC_UPDT'), 'path_master_resultable': (MASTER_RESULT, 'ADC_UPDT'), 'mill_order_catalog': (ORDER_CATALOG, 'ADC_UPDT'), 'mill_code_value': (CODE_VALUE, 'ADC_UPDT'), 'pathology_test_concept_map': (TEST_MAP, 'ADC_UPDT'), 'pathology_result_concept_map': (RESULT_MAP, 'ADC_UPDT'), 'pathology_unit_map': (UNIT_MAP, None)}
SOURCE_LOOKBACK = timedelta(days=2)
NATIVE_MIN_SUPPORT = 5
ENABLE_EMBED_LOOP = True
ENABLE_LIQUID_CLUSTERING = True
NUMERIC_REGEX = '^\\s*(?:<=|>=|<|>|≤|≥|=)?\\s*[+-]?(?:[0-9]+(?:[.][0-9]*)?|[.][0-9]+)(?:[eE][+-]?[0-9]+)?\\s*$'
RESULT_NORMALIZE_SQL = "LOWER(TRIM(REGEXP_REPLACE(result_txt,'\\\\s+',' ')))"
TEST_TIERS = ('curated', 'auto_high', 'auto_low')
RESULT_TIERS = ('curated', 'auto_high', 'auto_low', 'auto_anchor', 'auto_value', 'auto_genpos')

def _qn(name: str) -> str:
    """Quote a multipart table name, including catalogs that begin with a digit."""
    tick = chr(96)
    return '.'.join((f'{tick}{part}{tick}' for part in name.split('.')))

def _table_exists(name: str) -> bool:
    return spark.catalog.tableExists(name)

def _column_names(name: str) -> set[str]:
    if not _table_exists(name):
        return set()
    return {field.name for field in spark.table(name).schema.fields}

def _sql_string(value: str | None) -> str:
    if value is None:
        return 'NULL'
    return "'" + value.replace('\\', '\\\\').replace("'", "''") + "'"

def _ts_literal(value: datetime | None) -> str:
    if value is None:
        value = datetime(1980, 1, 1)
    if value.tzinfo is not None:
        value = value.astimezone(timezone.utc).replace(tzinfo=None)
    return "TIMESTAMP'" + value.strftime('%Y-%m-%d %H:%M:%S.%f') + "'"

def _empty_like(table_name: str) -> DataFrame:
    return spark.createDataFrame([], spark.table(table_name).schema)

def _union_distinct(frames: list[DataFrame], columns: list[str]) -> DataFrame:
    usable = [df.select(*columns) for df in frames if df is not None]
    if not usable:
        schema = StructType([StructField(c, StringType(), True) for c in columns])
        return spark.createDataFrame([], schema)
    return reduce(lambda left, right: left.unionByName(right), usable).dropDuplicates(columns)

def _latest_delta_version(table_name: str) -> int:
    return int(spark.sql(f'DESCRIBE HISTORY {_qn(table_name)} LIMIT 1').first()['version'])

def _max_timestamp(table_name: str, column_name: str | None) -> datetime | None:
    if column_name is None:
        return None
    row = spark.table(table_name).select(F.max(F.col(column_name)).alias('max_ts')).first()
    return row['max_ts'] if row else None

def _ensure_control_tables() -> None:
    spark.sql(f'\n        CREATE TABLE IF NOT EXISTS {_qn(MP_STATE)} (\n          source_name STRING NOT NULL,\n          table_name STRING NOT NULL,\n          last_delta_version BIGINT,\n          last_adc_updt TIMESTAMP,\n          last_success_at TIMESTAMP,\n          pipeline_version STRING\n        ) USING DELTA\n        ')
    spark.sql(f'\n        CREATE TABLE IF NOT EXISTS {_qn(MP_RUN_LOG)} (\n          run_id STRING,\n          started_at TIMESTAMP,\n          completed_at TIMESTAMP,\n          mode STRING,\n          status STRING,\n          pipeline_version STRING,\n          source_parent_count BIGINT,\n          staged_row_count BIGINT,\n          inserted_or_updated_rows BIGINT,\n          stale_rows_deleted BIGINT,\n          additive_map_keys BIGINT,\n          correction_map_keys BIGINT,\n          message STRING\n        ) USING DELTA\n        ')
    spark.sql(f'\n        CREATE TABLE IF NOT EXISTS {_qn(MP_FLAG)}\n        (id INT, rebuild_flagged BOOLEAN)\n        USING DELTA\n        ')
    spark.sql(f'\n        MERGE INTO {_qn(MP_FLAG)} t\n        USING (SELECT 1 AS id, FALSE AS rebuild_flagged) s\n        ON t.id=s.id\n        WHEN NOT MATCHED THEN INSERT *\n        ')

def _read_state() -> dict[str, dict]:
    if not _table_exists(MP_STATE):
        return {}
    return {row['source_name']: row.asDict() for row in spark.table(MP_STATE).collect()}

def _capture_cutoffs() -> dict[str, dict]:
    cutoffs = {}
    for source_name, (table_name, ts_col) in SOURCE_TABLES.items():
        cutoffs[source_name] = {'source_name': source_name, 'table_name': table_name, 'end_version': _latest_delta_version(table_name), 'end_adc_updt': _max_timestamp(table_name, ts_col), 'timestamp_column': ts_col}
    return cutoffs

def _read_changes(source_name: str, state: dict[str, dict], cutoffs: dict[str, dict]) -> tuple[DataFrame, str]:
    """
    Read source changes up to the run-start cutoff.

    Preferred path is CDF by Delta version, which retains delete keys. If CDF
    history is unavailable, fall back to a per-source ADC watermark with a
    two-day lookback. Tables without an ADC column fall back to a full-table
    key refresh when their Delta version changes.
    """
    cfg = cutoffs[source_name]
    table_name = cfg['table_name']
    end_version = cfg['end_version']
    ts_col = cfg['timestamp_column']
    previous = state.get(source_name)
    if previous is None or previous.get('last_delta_version') is None:
        return (_empty_like(table_name), 'seed')
    start_version = int(previous['last_delta_version']) + 1
    if start_version > end_version:
        return (_empty_like(table_name), 'no_change')
    try:
        changed = spark.read.format('delta').option('readChangeFeed', 'true').option('startingVersion', start_version).option('endingVersion', end_version).table(table_name).filter(F.col('_change_type') != F.lit('update_preimage'))
        return (changed, 'cdf')
    except Exception as exc:
        print(f'[map_pathology_v2] CDF fallback for {table_name}: {str(exc).splitlines()[0]}')
    if ts_col is not None:
        previous_ts = previous.get('last_adc_updt') or datetime(1980, 1, 1)
        start_ts = previous_ts - SOURCE_LOOKBACK
        end_ts = cfg['end_adc_updt']
        changed = spark.table(table_name).filter(F.col(ts_col) > F.lit(start_ts))
        if end_ts is not None:
            changed = changed.filter(F.col(ts_col) <= F.lit(end_ts))
        return (changed, 'timestamp')
    return (spark.table(table_name), 'full_key_refresh')

def _advance_state(cutoffs: dict[str, dict], source_names: list[str] | None=None) -> None:
    now = datetime.now(timezone.utc).replace(tzinfo=None)
    selected = source_names or list(cutoffs)
    rows = [(name, cutoffs[name]['table_name'], int(cutoffs[name]['end_version']), cutoffs[name]['end_adc_updt'], now, MP_VERSION) for name in selected]
    schema = 'source_name STRING, table_name STRING, last_delta_version BIGINT, last_adc_updt TIMESTAMP, last_success_at TIMESTAMP, pipeline_version STRING'
    updates = spark.createDataFrame(rows, schema)
    DeltaTable.forName(spark, MP_STATE).alias('t').merge(updates.alias('s'), 't.source_name=s.source_name').whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

def _set_rebuild_flag(value: bool) -> None:
    spark.sql(f'UPDATE {_qn(MP_FLAG)} SET rebuild_flagged={str(value).upper()} WHERE id=1')

def _read_rebuild_flag() -> bool:
    row = spark.table(MP_FLAG).where('id=1').select('rebuild_flagged').first()
    return bool(row['rebuild_flagged']) if row else False

def _write_run_log(**values) -> None:
    ordered = ['run_id', 'started_at', 'completed_at', 'mode', 'status', 'pipeline_version', 'source_parent_count', 'staged_row_count', 'inserted_or_updated_rows', 'stale_rows_deleted', 'additive_map_keys', 'correction_map_keys', 'message']
    row = tuple((values.get(col) for col in ordered))
    schema = 'run_id STRING, started_at TIMESTAMP, completed_at TIMESTAMP, mode STRING, status STRING, pipeline_version STRING, source_parent_count BIGINT, staged_row_count BIGINT, inserted_or_updated_rows BIGINT, stale_rows_deleted BIGINT, additive_map_keys BIGINT, correction_map_keys BIGINT, message STRING'
    spark.createDataFrame([row], schema).write.mode('append').saveAsTable(MP_RUN_LOG)
NEW_COLUMN_COMMENTS = {'source_system': 'Explicit source system: CERNER or TFC_LIMS.', 'source_parent_key': 'Stable reconciliation scope: linked EVENT_ID or raw LIMSNo+LabNo.', 'source_record_key': 'Stable row identity used for MERGE and stale-row reconciliation.', 'source_adc_updt': 'Greatest contributing source update timestamp; never changed by mapping-only backfills.', 'loaded_at': 'Timestamp at which this bronze row was written.', 'mapping_updated_at': 'Timestamp at which mapping columns were evaluated.', 'source_payload_hash': 'Hash of source/provenance fields used for complete change detection.', 'mapping_payload_hash': 'Hash of mapped/derived fields used for complete mapping change detection.', 'LIMSNo': 'TFC/LIMS source-system identifier. Combined with LabNo for raw specimen identity.', 'source_sequence_start': 'Minimum TFC result sequence in the assembled raw result island.', 'source_sequence_end': 'Maximum TFC result sequence in the assembled raw result island.', 'source_line_count': 'Number of source result-level lines assembled into this row.', 'person_id_mrn': 'PERSON_ID candidate resolved from MRN when the alias is unambiguous.', 'person_id_nhs': 'PERSON_ID candidate resolved from NHS number when the alias is unambiguous.', 'person_match_status': 'native, agreed, mrn_only, nhs_only, conflict, ambiguous, or unresolved.', 'person_match_conflict': 'TRUE when MRN and NHS resolve to different PERSON_ID values.', 'measurement_datetime_source': 'Source field selected for measurement_datetime.', 'test_mapping_match_type': 'exact_context, native_event_cd, native_nlmc, safe_code, or unmapped.', 'result_mapping_match_type': 'exact_context, native_context, safe_code_result, or unmapped.', 'unit_mapping_match_type': 'exact, normalized, or unmapped.', 'result_parse_status': 'blank, numeric, datetime, or text.', 'value_as_datetime': 'Conservatively parsed date/time result; raw text remains in value_source_value.', 'data_quality_flags': 'Pipe-delimited non-filtering quality indicators.', 'reference_nbr': 'Full Cerner reference number; lab_no remains the legacy 11-character projection.', 'range_low_raw': 'Unmodified source normal-low text.', 'range_high_raw': 'Unmodified source normal-high text.'}

def _apply_table_metadata() -> None:
    spark.sql(f"\n        ALTER TABLE {_qn(MP_TARGET)} SET TBLPROPERTIES (\n          'delta.enableChangeDataFeed'='true',\n          'delta.enableRowTracking'='true',\n          'delta.autoOptimize.optimizeWrite'='true',\n          'delta.autoOptimize.autoCompact'='true',\n          'delta.parquet.compression.codec'='zstd',\n          'comment'='Pathology foundation table v2. Correct LIMS specimen grain, complete source provenance, deterministic source reconciliation, and OMOP-aligned mappings without row-quality filters.'\n        )\n        ")
    existing = _column_names(MP_TARGET)
    for column_name, comment in NEW_COLUMN_COMMENTS.items():
        if column_name not in existing:
            continue
        escaped = comment.replace('\\', '\\\\').replace("'", "''")
        spark.sql(f"ALTER TABLE {_qn(MP_TARGET)} ALTER COLUMN {_qn(column_name)} COMMENT '{escaped}'")
    if ENABLE_LIQUID_CLUSTERING and {'source_table', 'source_parent_key'}.issubset(existing):
        try:
            spark.sql(f'ALTER TABLE {_qn(MP_TARGET)} CLUSTER BY (source_table, source_parent_key)')
        except Exception as exc:
            print(f'[map_pathology_v2] clustering note: {str(exc).splitlines()[0]}')

def _target_is_v2() -> bool:
    required = {'source_parent_key', 'source_record_key', 'source_adc_updt', 'source_payload_hash', 'mapping_payload_hash', 'LIMSNo', 'person_match_conflict'}
    return _table_exists(MP_TARGET) and required.issubset(_column_names(MP_TARGET))

def _prepare_incremental_scope(state: dict[str, dict], cutoffs: dict[str, dict]) -> tuple[DataFrame, dict[str, str]]:
    """
    Build touched parent scopes from every contributing source.

    The target is deliberately used as a compact source-to-parent index for
    order/catalog/code/master/map changes. Direct clinical-event, result-level,
    and sample-level CDF rows retain keys for deletes.
    """
    if not _target_is_v2():
        raise RuntimeError('Incremental scope requires a v2 target; run a full rebuild first.')
    changes: dict[str, DataFrame] = {}
    modes: dict[str, str] = {}
    for source_name in SOURCE_TABLES:
        changes[source_name], modes[source_name] = _read_changes(source_name, state, cutoffs)
        print(f'[map_pathology_v2] {source_name}: {modes[source_name]}')
    target = spark.table(MP_TARGET)
    linked_target = target.filter(F.col('source_table') == 'linked')
    raw_target = target.filter(F.col('source_table') == 'raw')
    linked_ids: list[DataFrame] = []
    raw_keys: list[DataFrame] = []
    linked_ids.append(changes['mill_clinical_event'].select(F.col('EVENT_ID').cast('long').alias('EVENT_ID')).filter(F.col('EVENT_ID').isNotNull()))
    raw_keys.append(changes['path_patient_resultlevel'].select(F.col('LIMSNo').cast('int').alias('LIMSNo'), F.col('LabNo').cast('string').alias('LabNo')))
    raw_keys.append(changes['path_patient_samplelevel'].select(F.col('LIMSNo').cast('int').alias('LIMSNo'), F.col('LabNo').cast('string').alias('LabNo')))
    changed_orders = changes['mill_orders'].select(F.col('ORDER_ID').cast('long').alias('ORDER_ID')).filter(F.col('ORDER_ID').isNotNull()).dropDuplicates()
    linked_ids.append(linked_target.join(changed_orders, linked_target.order_id == changed_orders.ORDER_ID, 'inner').select(F.col('source_event_id').cast('long').alias('EVENT_ID')))
    alias_changes = changes['mill_person_alias'].select(F.col('PERSON_ID').cast('long').alias('PERSON_ID'), F.col('PERSON_ALIAS_TYPE_CD').cast('int').alias('PERSON_ALIAS_TYPE_CD'), F.col('ALIAS').cast('string').alias('ALIAS'))
    changed_people = alias_changes.select('PERSON_ID').filter(F.col('PERSON_ID').isNotNull()).dropDuplicates()
    linked_ids.append(linked_target.join(changed_people, 'PERSON_ID', 'inner').select(F.col('source_event_id').cast('long').alias('EVENT_ID')))
    changed_mrn = alias_changes.filter(F.col('PERSON_ALIAS_TYPE_CD') == 10).select(F.col('ALIAS').alias('_changed_mrn')).filter(F.col('_changed_mrn').isNotNull()).dropDuplicates()
    changed_nhs = alias_changes.filter(F.col('PERSON_ALIAS_TYPE_CD') == 18).select(F.col('ALIAS').alias('_changed_nhs')).filter(F.col('_changed_nhs').isNotNull()).dropDuplicates()
    sample_index = spark.table(SAMPLE_LEVEL).select(F.col('LIMSNo').cast('int').alias('LIMSNo'), F.col('LabNo').cast('string').alias('LabNo'), F.col('MRN').cast('string').alias('MRN'), F.col('NHSNo').cast('string').alias('NHSNo'))
    raw_keys.append(sample_index.join(changed_mrn, sample_index.MRN == changed_mrn._changed_mrn, 'inner').select('LIMSNo', 'LabNo'))
    raw_keys.append(sample_index.join(changed_nhs, sample_index.NHSNo == changed_nhs._changed_nhs, 'inner').select('LIMSNo', 'LabNo'))
    changed_catalogs = changes['mill_order_catalog'].select(F.col('CATALOG_CD').cast('long').alias('CATALOG_CD')).filter(F.col('CATALOG_CD').isNotNull()).dropDuplicates()
    linked_ids.append(linked_target.join(changed_catalogs, linked_target.catalog_cd == changed_catalogs.CATALOG_CD, 'inner').select(F.col('source_event_id').cast('long').alias('EVENT_ID')))
    changed_codes = changes['mill_code_value'].select(F.col('CODE_VALUE').cast('long').alias('CODE_VALUE')).filter(F.col('CODE_VALUE').isNotNull()).dropDuplicates()
    linked_ids.append(linked_target.join(changed_codes, (linked_target.EVENT_CD == changed_codes.CODE_VALUE) | (linked_target.result_units_cd == changed_codes.CODE_VALUE) | (linked_target.normalcy_cd == changed_codes.CODE_VALUE), 'inner').select(F.col('source_event_id').cast('long').alias('EVENT_ID')))
    changed_master = changes['path_master_resultable'].select(F.col('WkgCode').cast('string').alias('_master_wkg'), F.col('TFCCode').cast('string').alias('_master_tfc')).dropDuplicates()
    raw_keys.append(raw_target.join(changed_master, raw_target.WkgCode.eqNullSafe(changed_master._master_wkg) & raw_target.code.eqNullSafe(changed_master._master_tfc), 'inner').select('LIMSNo', F.col('lab_no').alias('LabNo')))
    changed_test = changes['pathology_test_concept_map'].select(F.col('code_system').alias('_tm_system'), F.col('code').alias('_tm_code'), F.col('description').alias('_tm_description')).dropDuplicates()
    test_affected = target.join(changed_test, (target.code_system == changed_test._tm_system) & target.code.eqNullSafe(changed_test._tm_code), 'inner')
    linked_ids.append(test_affected.filter(F.col('source_table') == 'linked').select(F.col('source_event_id').cast('long').alias('EVENT_ID')))
    raw_keys.append(test_affected.filter(F.col('source_table') == 'raw').select('LIMSNo', F.col('lab_no').alias('LabNo')))
    changed_result = changes['pathology_result_concept_map'].select(F.col('code_system').alias('_rm_system'), F.col('code').alias('_rm_code'), F.col('description').alias('_rm_description'), F.col('result_normalized').alias('_rm_result')).dropDuplicates()
    target_with_result = target.withColumn('_target_result_normalized', F.lower(F.trim(F.regexp_replace(F.col('value_source_value'), '\\s+', ' '))))
    result_affected = target_with_result.join(changed_result, (target_with_result.code_system == changed_result._rm_system) & target_with_result.code.eqNullSafe(changed_result._rm_code) & target_with_result._target_result_normalized.eqNullSafe(changed_result._rm_result), 'inner')
    linked_ids.append(result_affected.filter(F.col('source_table') == 'linked').select(F.col('source_event_id').cast('long').alias('EVENT_ID')))
    raw_keys.append(result_affected.filter(F.col('source_table') == 'raw').select('LIMSNo', F.col('lab_no').alias('LabNo')))
    changed_units = changes['pathology_unit_map'].select(F.lower(F.trim(F.col('unit_source_value'))).alias('_unit_norm')).dropDuplicates()
    unit_affected = target.withColumn('_target_unit_norm', F.lower(F.trim(F.col('unit_source_value')))).join(changed_units, F.col('_target_unit_norm').eqNullSafe(changed_units._unit_norm), 'inner')
    linked_ids.append(unit_affected.filter(F.col('source_table') == 'linked').select(F.col('source_event_id').cast('long').alias('EVENT_ID')))
    raw_keys.append(unit_affected.filter(F.col('source_table') == 'raw').select('LIMSNo', F.col('lab_no').alias('LabNo')))
    native_test = spark.table(MP_NATIVE_TEST).select(F.col('key_type').alias('_nt_type'), F.col('key_value').alias('_nt_value')).dropDuplicates()
    native_test_affected = target.filter(F.col('measurement_concept_id').isNull()).join(native_test, (F.col('source_table') == 'linked') & (F.col('_nt_type') == 'EVENT_CD') & (F.col('EVENT_CD').cast('string') == F.col('_nt_value')) | (F.col('source_table') == 'raw') & (F.col('_nt_type') == 'NLMC_ID') & F.col('nlmc_id').eqNullSafe(F.col('_nt_value')), 'inner')
    linked_ids.append(native_test_affected.filter(F.col('source_table') == 'linked').select(F.col('source_event_id').cast('long').alias('EVENT_ID')))
    raw_keys.append(native_test_affected.filter(F.col('source_table') == 'raw').select('LIMSNo', F.col('lab_no').alias('LabNo')))
    native_result = spark.table(MP_NATIVE_RESULT).select(F.col('key_type').alias('_nr_type'), F.col('key_value').alias('_nr_value'), F.col('result_normalized').alias('_nr_result')).dropDuplicates()
    native_result_affected = target_with_result.filter(F.col('value_as_concept_id').isNull() & ~F.col('result_status').isin('numeric', 'datetime', 'missing')).join(native_result, (F.col('source_table') == 'linked') & (F.col('_nr_type') == 'EVENT_CD') & (F.col('EVENT_CD').cast('string') == F.col('_nr_value')) & F.col('_target_result_normalized').eqNullSafe(F.col('_nr_result')) | (F.col('source_table') == 'raw') & (F.col('_nr_type') == 'NLMC_ID') & F.col('nlmc_id').eqNullSafe(F.col('_nr_value')) & F.col('_target_result_normalized').eqNullSafe(F.col('_nr_result')), 'inner')
    linked_ids.append(native_result_affected.filter(F.col('source_table') == 'linked').select(F.col('source_event_id').cast('long').alias('EVENT_ID')))
    raw_keys.append(native_result_affected.filter(F.col('source_table') == 'raw').select('LIMSNo', F.col('lab_no').alias('LabNo')))
    linked_scope = reduce(lambda left, right: left.unionByName(right), linked_ids).filter(F.col('EVENT_ID').isNotNull()).dropDuplicates(['EVENT_ID'])
    raw_scope = reduce(lambda left, right: left.unionByName(right), raw_keys).dropDuplicates(['LIMSNo', 'LabNo'])
    linked_scope.createOrReplaceTempView('mp_v2_linked_scope')
    raw_scope.createOrReplaceTempView('mp_v2_raw_scope')
    linked_parents = linked_scope.select(F.lit('linked').alias('source_table'), F.concat(F.lit('linked|'), F.col('EVENT_ID').cast('string')).alias('source_parent_key'), F.col('EVENT_ID').cast('long').alias('source_event_id'), F.lit(None).cast('int').alias('LIMSNo'), F.lit(None).cast('string').alias('lab_no'))
    raw_parents = raw_scope.select(F.lit('raw').alias('source_table'), F.concat_ws('|', F.lit('raw'), F.coalesce(F.col('LIMSNo').cast('string'), F.lit('∅')), F.coalesce(F.col('LabNo'), F.lit('∅'))).alias('source_parent_key'), F.lit(None).cast('long').alias('source_event_id'), F.col('LIMSNo').cast('int').alias('LIMSNo'), F.col('LabNo').cast('string').alias('lab_no'))
    parents = linked_parents.unionByName(raw_parents).dropDuplicates(['source_parent_key'])
    return (parents, modes)

def _scope_from_parent_keys(parents: DataFrame) -> DataFrame:
    """Register source scope views from a target-derived set of parent keys."""
    target_keys = spark.table(MP_TARGET).select('source_parent_key', 'source_table', 'source_event_id', 'LIMSNo', 'lab_no').dropDuplicates(['source_parent_key'])
    expanded = target_keys.join(parents.select('source_parent_key').dropDuplicates(), 'source_parent_key', 'inner')
    expanded.filter(F.col('source_table') == 'linked').select(F.col('source_event_id').cast('long').alias('EVENT_ID')).dropDuplicates().createOrReplaceTempView('mp_v2_linked_scope')
    expanded.filter(F.col('source_table') == 'raw').select(F.col('LIMSNo').cast('int').alias('LIMSNo'), F.col('lab_no').cast('string').alias('LabNo')).dropDuplicates().createOrReplaceTempView('mp_v2_raw_scope')
    return expanded.select('source_table', 'source_parent_key', 'source_event_id', 'LIMSNo', 'lab_no').dropDuplicates(['source_parent_key'])

def _refresh_native_crosswalks() -> None:
    """
    Materialize conservative native-key crosswalks from already accepted mappings.

    Only curated/auto_high evidence is used, a native key must point to exactly
    one concept, and it must have at least NATIVE_MIN_SUPPORT supporting rows.
    This prevents the fallback from turning a context collision into a mapping.
    """
    if not _table_exists(MP_TARGET):
        spark.sql(f'\n            CREATE OR REPLACE TABLE {_qn(MP_NATIVE_TEST)} (\n              key_type STRING, key_value STRING, measurement_concept_id BIGINT,\n              concept_name STRING, confidence_tier STRING, support_count BIGINT\n            ) USING DELTA\n            ')
        spark.sql(f'\n            CREATE OR REPLACE TABLE {_qn(MP_NATIVE_RESULT)} (\n              key_type STRING, key_value STRING, result_normalized STRING,\n              value_as_concept_id BIGINT, concept_name STRING,\n              confidence_tier STRING, support_count BIGINT\n            ) USING DELTA\n            ')
        return
    spark.sql(f"\n        CREATE OR REPLACE TABLE {_qn(MP_NATIVE_TEST)} USING DELTA AS\n        WITH evidence AS (\n          SELECT 'EVENT_CD' AS key_type, CAST(EVENT_CD AS STRING) AS key_value,\n                 measurement_concept_id AS cid, COUNT(*) AS support_count\n          FROM {_qn(MP_TARGET)}\n          WHERE source_table='linked' AND EVENT_CD IS NOT NULL\n            AND measurement_concept_id IS NOT NULL\n            AND test_confidence_tier IN ('curated','auto_high')\n          GROUP BY EVENT_CD, measurement_concept_id\n          UNION ALL\n          SELECT 'NLMC_ID', nlmc_id, measurement_concept_id, COUNT(*)\n          FROM {_qn(MP_TARGET)}\n          WHERE source_table='raw' AND nlmc_id IS NOT NULL AND TRIM(nlmc_id)<>''\n            AND measurement_concept_id IS NOT NULL\n            AND test_confidence_tier IN ('curated','auto_high')\n          GROUP BY nlmc_id, measurement_concept_id\n        ),\n        safe AS (\n          SELECT key_type, key_value, MIN(cid) AS cid, SUM(support_count) AS support_count\n          FROM evidence\n          GROUP BY key_type, key_value\n          HAVING COUNT(DISTINCT cid)=1 AND SUM(support_count)>={NATIVE_MIN_SUPPORT}\n        )\n        SELECT s.key_type, s.key_value, s.cid AS measurement_concept_id,\n               c.concept_name, 'native_safe' AS confidence_tier, s.support_count\n        FROM safe s\n        LEFT JOIN {_qn(CONCEPT)} c ON c.concept_id=s.cid\n        ")
    spark.sql(f"\n        CREATE OR REPLACE TABLE {_qn(MP_NATIVE_RESULT)} USING DELTA AS\n        WITH evidence AS (\n          SELECT 'EVENT_CD' AS key_type, CAST(EVENT_CD AS STRING) AS key_value,\n                 LOWER(TRIM(REGEXP_REPLACE(value_source_value,'\\\\s+',' '))) AS result_normalized,\n                 value_as_concept_id AS cid, COUNT(*) AS support_count\n          FROM {_qn(MP_TARGET)}\n          WHERE source_table='linked' AND EVENT_CD IS NOT NULL\n            AND value_as_concept_id IS NOT NULL\n            AND result_confidence_tier IN ('curated','auto_high')\n          GROUP BY EVENT_CD,\n                   LOWER(TRIM(REGEXP_REPLACE(value_source_value,'\\\\s+',' '))),\n                   value_as_concept_id\n          UNION ALL\n          SELECT 'NLMC_ID', nlmc_id,\n                 LOWER(TRIM(REGEXP_REPLACE(value_source_value,'\\\\s+',' '))),\n                 value_as_concept_id, COUNT(*)\n          FROM {_qn(MP_TARGET)}\n          WHERE source_table='raw' AND nlmc_id IS NOT NULL AND TRIM(nlmc_id)<>''\n            AND value_as_concept_id IS NOT NULL\n            AND result_confidence_tier IN ('curated','auto_high')\n          GROUP BY nlmc_id,\n                   LOWER(TRIM(REGEXP_REPLACE(value_source_value,'\\\\s+',' '))),\n                   value_as_concept_id\n        ),\n        safe AS (\n          SELECT key_type, key_value, result_normalized,\n                 MIN(cid) AS cid, SUM(support_count) AS support_count\n          FROM evidence\n          GROUP BY key_type, key_value, result_normalized\n          HAVING COUNT(DISTINCT cid)=1 AND SUM(support_count)>={NATIVE_MIN_SUPPORT}\n        )\n        SELECT s.key_type, s.key_value, s.result_normalized,\n               s.cid AS value_as_concept_id, c.concept_name,\n               'native_safe' AS confidence_tier, s.support_count\n        FROM safe s\n        LEFT JOIN {_qn(CONCEPT)} c ON c.concept_id=s.cid\n        ")

def _exclusion_regex_sql() -> str:
    patterns = [row['pattern'] for row in spark.table(EXCL_TBL).select('pattern').where('pattern IS NOT NULL').collect()]
    if not patterns:
        return '(?!)'
    combined = '(' + '|'.join(patterns) + ')'
    return combined.replace('\\', '\\\\').replace("'", "''")

def _tier_sql(values: tuple[str, ...]) -> str:
    return '(' + ','.join((_sql_string(v) for v in values)) + ')'

def _mp_build_select(full: bool) -> str:
    """
    Return the complete source-to-target projection.

    Incremental scope views contain whole linked EVENT_ID parents and whole raw
    (LIMSNo, LabNo) parents, so reassembly and stale-row reconciliation are safe.
    """
    linked_scope_join = '' if full else 'INNER JOIN mp_v2_linked_scope scope ON scope.EVENT_ID=ce.EVENT_ID'
    raw_scope_result_join = '' if full else 'INNER JOIN mp_v2_raw_scope scope ON scope.LIMSNo <=> r.LIMSNo AND scope.LabNo <=> r.LabNo'
    raw_scope_sample_join = '' if full else 'INNER JOIN mp_v2_raw_scope scope ON scope.LIMSNo <=> sl.LIMSNo AND scope.LabNo <=> sl.LabNo'
    numeric_re = NUMERIC_REGEX.replace('\\', '\\\\').replace("'", "''")
    exclusion_re = _exclusion_regex_sql()
    test_tiers = _tier_sql(TEST_TIERS)
    result_tiers = _tier_sql(RESULT_TIERS)
    return f"\n    WITH\n    pa_person_mrn AS (\n      SELECT PERSON_ID, ALIAS AS canonical_mrn, ADC_UPDT AS canonical_mrn_adc\n      FROM (\n        SELECT CAST(PERSON_ID AS BIGINT) AS PERSON_ID, ALIAS, ADC_UPDT,\n               ROW_NUMBER() OVER (\n                 PARTITION BY PERSON_ID\n                 ORDER BY BEG_EFFECTIVE_DT_TM DESC NULLS LAST,\n                          PERSON_ALIAS_ID DESC NULLS LAST\n               ) rn\n        FROM {_qn(PERSON_ALIAS)}\n        WHERE ACTIVE_IND=1 AND PERSON_ALIAS_TYPE_CD=10\n      ) WHERE rn=1\n    ),\n    pa_person_nhs AS (\n      SELECT PERSON_ID, ALIAS AS canonical_nhs, ADC_UPDT AS canonical_nhs_adc\n      FROM (\n        SELECT CAST(PERSON_ID AS BIGINT) AS PERSON_ID, ALIAS, ADC_UPDT,\n               ROW_NUMBER() OVER (\n                 PARTITION BY PERSON_ID\n                 ORDER BY BEG_EFFECTIVE_DT_TM DESC NULLS LAST,\n                          PERSON_ALIAS_ID DESC NULLS LAST\n               ) rn\n        FROM {_qn(PERSON_ALIAS)}\n        WHERE ACTIVE_IND=1 AND PERSON_ALIAS_TYPE_CD=18\n      ) WHERE rn=1\n    ),\n    mrn_alias_stats AS (\n      SELECT ALIAS, COUNT(DISTINCT PERSON_ID) AS person_count\n      FROM {_qn(PERSON_ALIAS)}\n      WHERE ACTIVE_IND=1 AND PERSON_ALIAS_TYPE_CD=10\n      GROUP BY ALIAS\n    ),\n    mrn_alias_latest AS (\n      SELECT ALIAS, CAST(PERSON_ID AS BIGINT) AS PERSON_ID, ADC_UPDT\n      FROM (\n        SELECT ALIAS, PERSON_ID, ADC_UPDT,\n               ROW_NUMBER() OVER (\n                 PARTITION BY ALIAS\n                 ORDER BY BEG_EFFECTIVE_DT_TM DESC NULLS LAST,\n                          PERSON_ALIAS_ID DESC NULLS LAST\n               ) rn\n        FROM {_qn(PERSON_ALIAS)}\n        WHERE ACTIVE_IND=1 AND PERSON_ALIAS_TYPE_CD=10\n      ) WHERE rn=1\n    ),\n    mrn_resolver AS (\n      SELECT l.ALIAS,\n             CASE WHEN s.person_count=1 THEN l.PERSON_ID END AS PERSON_ID,\n             s.person_count, l.ADC_UPDT\n      FROM mrn_alias_latest l\n      JOIN mrn_alias_stats s USING (ALIAS)\n    ),\n    nhs_alias_stats AS (\n      SELECT ALIAS, COUNT(DISTINCT PERSON_ID) AS person_count\n      FROM {_qn(PERSON_ALIAS)}\n      WHERE ACTIVE_IND=1 AND PERSON_ALIAS_TYPE_CD=18\n      GROUP BY ALIAS\n    ),\n    nhs_alias_latest AS (\n      SELECT ALIAS, CAST(PERSON_ID AS BIGINT) AS PERSON_ID, ADC_UPDT\n      FROM (\n        SELECT ALIAS, PERSON_ID, ADC_UPDT,\n               ROW_NUMBER() OVER (\n                 PARTITION BY ALIAS\n                 ORDER BY BEG_EFFECTIVE_DT_TM DESC NULLS LAST,\n                          PERSON_ALIAS_ID DESC NULLS LAST\n               ) rn\n        FROM {_qn(PERSON_ALIAS)}\n        WHERE ACTIVE_IND=1 AND PERSON_ALIAS_TYPE_CD=18\n      ) WHERE rn=1\n    ),\n    nhs_resolver AS (\n      SELECT l.ALIAS,\n             CASE WHEN s.person_count=1 THEN l.PERSON_ID END AS PERSON_ID,\n             s.person_count, l.ADC_UPDT\n      FROM nhs_alias_latest l\n      JOIN nhs_alias_stats s USING (ALIAS)\n    ),\n    linked_ranked AS (\n      SELECT\n        ce.CLINICAL_EVENT_ID, ce.EVENT_ID, ce.PERSON_ID, ce.ENCNTR_ID,\n        ce.ORDER_ID, ce.CATALOG_CD, ce.EVENT_CD, ce.PARENT_EVENT_ID,\n        ce.EVENT_RELTN_CD, ce.VALID_FROM_DT_TM, ce.VALID_UNTIL_DT_TM,\n        ce.EVENT_START_DT_TM, ce.EVENT_END_DT_TM, ce.PERFORMED_DT_TM,\n        ce.VERIFIED_DT_TM, ce.RESULT_VAL, ce.RESULT_UNITS_CD,\n        ce.NORMAL_LOW, ce.NORMAL_HIGH, ce.NORMALCY_CD,\n        ce.RECORD_STATUS_CD, ce.RESULT_STATUS_CD, ce.AUTHENTIC_FLAG,\n        ce.CLINSIG_UPDT_DT_TM, ce.UPDT_CNT, ce.CONTRIBUTOR_SYSTEM_CD,\n        ce.REFERENCE_NBR, ce.EVENT_TITLE_TEXT, ce.EVENT_TAG,\n        ce.ADC_UPDT AS ce_adc,\n        o.ORDER_MNEMONIC, o.HNA_ORDER_MNEMONIC, o.ORDERED_AS_MNEMONIC,\n        o.ADC_UPDT AS order_adc,\n        oc.PRIMARY_MNEMONIC, oc.DESCRIPTION AS catalog_description,\n        oc.ADC_UPDT AS catalog_adc,\n        ROW_NUMBER() OVER (\n          PARTITION BY ce.EVENT_ID\n          ORDER BY ce.VALID_UNTIL_DT_TM DESC NULLS LAST,\n                   ce.UPDT_CNT DESC NULLS LAST,\n                   ce.CLINSIG_UPDT_DT_TM DESC NULLS LAST,\n                   ce.ADC_UPDT DESC NULLS LAST,\n                   ce.CLINICAL_EVENT_ID DESC NULLS LAST\n        ) AS rn\n      FROM {_qn(CE)} ce\n      {linked_scope_join}\n      LEFT JOIN {_qn(ORDERS)} o ON CAST(o.ORDER_ID AS BIGINT)=ce.ORDER_ID\n      LEFT JOIN {_qn(ORDER_CATALOG)} oc ON CAST(oc.CATALOG_CD AS BIGINT)=ce.CATALOG_CD\n      WHERE ce.EVENT_CLASS_CD IN (233,236)\n        AND COALESCE(CAST(oc.CATALOG_TYPE_CD AS BIGINT),\n                     CAST(o.CATALOG_TYPE_CD AS BIGINT))=2513\n    ),\n    linked_src AS (\n      SELECT\n        'linked' AS source_table,\n        'CERNER' AS source_system,\n        CONCAT('linked|', CAST(ce.EVENT_ID AS STRING)) AS source_parent_key,\n        CONCAT('linked|', CAST(ce.EVENT_ID AS STRING)) AS source_record_key,\n        CAST(ce.EVENT_ID AS BIGINT) AS source_event_id,\n        FALSE AS is_synthetic_key,\n        SUBSTRING(ce.REFERENCE_NBR,1,11) AS lab_no,\n        CAST(NULL AS INT) AS LIMSNo,\n        CAST(NULL AS BIGINT) AS source_sequence_start,\n        CAST(NULL AS BIGINT) AS source_sequence_end,\n        CAST(1 AS BIGINT) AS source_line_count,\n        CAST(NULL AS STRING) AS source_month_year,\n        CAST(NULL AS STRING) AS source_month_year_auth,\n        CAST(ce.PERSON_ID AS BIGINT) AS PERSON_ID,\n        CAST(ce.ENCNTR_ID AS BIGINT) AS ENCNTR_ID,\n        CAST(NULL AS BIGINT) AS person_id_mrn,\n        CAST(NULL AS BIGINT) AS person_id_nhs,\n        'native' AS person_match_status,\n        FALSE AS person_match_conflict,\n        pm.canonical_mrn AS MRN,\n        pn.canonical_nhs AS NHS_Number,\n        CAST(ce.EVENT_CD AS INT) AS EVENT_CD,\n        COALESCE(cv.DESCRIPTION,cv.DISPLAY,ce.EVENT_TITLE_TEXT,ce.EVENT_TAG) AS EVENT_CD_DISPLAY,\n        COALESCE(ce.EVENT_END_DT_TM,ce.PERFORMED_DT_TM,\n                 ce.EVENT_START_DT_TM,ce.VERIFIED_DT_TM) AS measurement_datetime,\n        CASE WHEN ce.EVENT_END_DT_TM IS NOT NULL THEN 'EVENT_END_DT_TM'\n             WHEN ce.PERFORMED_DT_TM IS NOT NULL THEN 'PERFORMED_DT_TM'\n             WHEN ce.EVENT_START_DT_TM IS NOT NULL THEN 'EVENT_START_DT_TM'\n             WHEN ce.VERIFIED_DT_TM IS NOT NULL THEN 'VERIFIED_DT_TM'\n        END AS measurement_datetime_source,\n        'CERNER_TESTCODE' AS code_system,\n        COALESCE(NULLIF(TRIM(ce.ORDER_MNEMONIC),''),\n                 NULLIF(TRIM(ce.HNA_ORDER_MNEMONIC),''),\n                 NULLIF(TRIM(ce.ORDERED_AS_MNEMONIC),''),\n                 NULLIF(TRIM(ce.PRIMARY_MNEMONIC),''),\n                 CAST(ce.EVENT_CD AS STRING)) AS code,\n        CASE WHEN NULLIF(TRIM(ce.ORDER_MNEMONIC),'') IS NOT NULL THEN 'ORDER_MNEMONIC'\n             WHEN NULLIF(TRIM(ce.HNA_ORDER_MNEMONIC),'') IS NOT NULL THEN 'HNA_ORDER_MNEMONIC'\n             WHEN NULLIF(TRIM(ce.ORDERED_AS_MNEMONIC),'') IS NOT NULL THEN 'ORDERED_AS_MNEMONIC'\n             WHEN NULLIF(TRIM(ce.PRIMARY_MNEMONIC),'') IS NOT NULL THEN 'PRIMARY_MNEMONIC'\n             ELSE 'EVENT_CD'\n        END AS code_source,\n        COALESCE(cv.DESCRIPTION,cv.DISPLAY,ce.EVENT_TITLE_TEXT,ce.EVENT_TAG,\n                 ce.catalog_description,CAST(ce.EVENT_CD AS STRING)) AS description,\n        CASE WHEN cv.DESCRIPTION IS NOT NULL THEN 'CODE_VALUE_DESCRIPTION'\n             WHEN cv.DISPLAY IS NOT NULL THEN 'CODE_VALUE_DISPLAY'\n             WHEN ce.EVENT_TITLE_TEXT IS NOT NULL THEN 'EVENT_TITLE_TEXT'\n             WHEN ce.EVENT_TAG IS NOT NULL THEN 'EVENT_TAG'\n             WHEN ce.catalog_description IS NOT NULL THEN 'ORDER_CATALOG_DESCRIPTION'\n             ELSE 'EVENT_CD'\n        END AS description_source,\n        ce.RESULT_VAL AS result_txt,\n        COALESCE(ucv.DESCRIPTION,ucv.DISPLAY) AS unit_source_value,\n        ce.NORMAL_LOW AS range_low_raw,\n        ce.NORMAL_HIGH AS range_high_raw,\n        TRY_CAST(ce.NORMAL_LOW AS DOUBLE) AS range_low,\n        TRY_CAST(ce.NORMAL_HIGH AS DOUBLE) AS range_high,\n        COALESCE(ncv.DESCRIPTION,ncv.DISPLAY) AS normalcy,\n        CAST(NULL AS STRING) AS WkgCode,\n        CAST(NULL AS STRING) AS nlmc_id,\n        COALESCE(ce.VERIFIED_DT_TM,ce.EVENT_END_DT_TM,ce.PERFORMED_DT_TM) AS ReportDate,\n        GREATEST(ce.ce_adc,ce.order_adc,ce.catalog_adc,cv.ADC_UPDT,\n                 ucv.ADC_UPDT,ncv.ADC_UPDT,pm.canonical_mrn_adc,pn.canonical_nhs_adc)\n          AS source_adc_updt,\n        ce.REFERENCE_NBR AS reference_nbr,\n        CAST(ce.CLINICAL_EVENT_ID AS BIGINT) AS clinical_event_id,\n        CAST(ce.ORDER_ID AS BIGINT) AS order_id,\n        CAST(ce.CATALOG_CD AS BIGINT) AS catalog_cd,\n        CAST(ce.PARENT_EVENT_ID AS BIGINT) AS parent_event_id,\n        CAST(ce.EVENT_RELTN_CD AS BIGINT) AS event_reltn_cd,\n        ce.VALID_FROM_DT_TM AS valid_from_dt_tm,\n        ce.VALID_UNTIL_DT_TM AS valid_until_dt_tm,\n        ce.EVENT_START_DT_TM AS event_start_dt_tm,\n        ce.EVENT_END_DT_TM AS event_end_dt_tm,\n        ce.PERFORMED_DT_TM AS performed_dt_tm,\n        ce.VERIFIED_DT_TM AS verified_dt_tm,\n        CAST(ce.RECORD_STATUS_CD AS BIGINT) AS record_status_cd,\n        CAST(ce.RESULT_STATUS_CD AS BIGINT) AS result_status_cd,\n        CAST(ce.AUTHENTIC_FLAG AS BIGINT) AS authentic_flag,\n        ce.CLINSIG_UPDT_DT_TM AS clinsig_updt_dt_tm,\n        CAST(ce.UPDT_CNT AS BIGINT) AS source_updt_cnt,\n        CAST(ce.CONTRIBUTOR_SYSTEM_CD AS BIGINT) AS contributor_system_cd,\n        CAST(ce.RESULT_UNITS_CD AS BIGINT) AS result_units_cd,\n        CAST(ce.NORMALCY_CD AS BIGINT) AS normalcy_cd,\n        CAST(NULL AS STRING) AS legacy_wkg_code,\n        CAST(NULL AS STRING) AS legacy_tfc_code,\n        CAST(NULL AS TIMESTAMP) AS request_dt,\n        CAST(NULL AS TIMESTAMP) AS sample_dt,\n        CAST(NULL AS TIMESTAMP) AS receipt_dt,\n        CAST(NULL AS TIMESTAMP) AS booked_in_dt,\n        CAST(NULL AS STRING) AS order_no,\n        CAST(NULL AS STRING) AS visit_id,\n        CAST(NULL AS STRING) AS ass_auth_code,\n        CAST(NULL AS STRING) AS body_site_code,\n        CAST(NULL AS STRING) AS specimen_type_code,\n        CAST(NULL AS STRING) AS specimen_category,\n        CAST(NULL AS STRING) AS urgent_flag,\n        CAST(NULL AS STRING) AS source_code,\n        CAST(NULL AS STRING) AS clinician_code,\n        CAST(NULL AS STRING) AS master_section_code,\n        CAST(NULL AS STRING) AS work_section_code,\n        CAST(NULL AS STRING) AS report_section,\n        CAST(NULL AS STRING) AS master_result_type,\n        CAST(NULL AS STRING) AS master_result_format,\n        CAST(NULL AS INT) AS master_num_val_upper,\n        CAST(NULL AS INT) AS master_num_val_dps\n      FROM linked_ranked ce\n      LEFT JOIN {_qn(CODE_VALUE)} cv ON CAST(cv.CODE_VALUE AS BIGINT)=ce.EVENT_CD\n      LEFT JOIN {_qn(CODE_VALUE)} ucv ON CAST(ucv.CODE_VALUE AS BIGINT)=ce.RESULT_UNITS_CD\n      LEFT JOIN {_qn(CODE_VALUE)} ncv ON CAST(ncv.CODE_VALUE AS BIGINT)=ce.NORMALCY_CD\n      LEFT JOIN pa_person_mrn pm ON pm.PERSON_ID=CAST(ce.PERSON_ID AS BIGINT)\n      LEFT JOIN pa_person_nhs pn ON pn.PERSON_ID=CAST(ce.PERSON_ID AS BIGINT)\n      WHERE ce.rn=1\n    ),\n    rl_pre AS (\n      SELECT\n        CAST(r.LIMSNo AS INT) AS LIMSNo,\n        r.LabNo,\n        COALESCE(NULLIF(TRIM(r.TFCCode),''),NULLIF(TRIM(r.LegTFCCode),'')) AS TFCCode,\n        COALESCE(NULLIF(TRIM(r.WkgCode),''),NULLIF(TRIM(r.LegWkgCode),'')) AS WkgCode,\n        r.LegTFCCode, r.LegWkgCode, CAST(r.TFCResultSeq AS BIGINT) AS TFCResultSeq,\n        r.TFCValue, r.MonthYear, r.MonthYearAuth, r.ADC_UPDT\n      FROM {_qn(RESULT_LEVEL)} r\n      {raw_scope_result_join}\n    ),\n    rl_island AS (\n      SELECT p.*,\n        CASE\n          WHEN TFCResultSeq IS NULL THEN\n            XXHASH64(CONCAT_WS('|',COALESCE(CAST(LIMSNo AS STRING),'∅'),\n              COALESCE(LabNo,'∅'),COALESCE(TFCCode,'∅'),COALESCE(WkgCode,'∅'),\n              COALESCE(TFCValue,'∅'),COALESCE(MonthYear,'∅')))\n          WHEN TFCCode RLIKE '^(INTER|UNU)' THEN TFCResultSeq\n          ELSE TFCResultSeq - DENSE_RANK() OVER (\n            PARTITION BY LIMSNo,LabNo,TFCCode,WkgCode\n            ORDER BY TFCResultSeq\n          )\n        END AS island_id\n      FROM rl_pre p\n    ),\n    rl AS (\n      SELECT\n        LIMSNo,LabNo,TFCCode,WkgCode,\n        MAX(LegTFCCode) AS LegTFCCode,\n        MAX(LegWkgCode) AS LegWkgCode,\n        MIN(TFCResultSeq) AS TFCResultSeq,\n        MAX(TFCResultSeq) AS TFCResultSeqEnd,\n        COUNT(*) AS source_line_count,\n        MIN(MonthYear) AS MonthYear,\n        MIN(MonthYearAuth) AS MonthYearAuth,\n        CONCAT_WS(\n          '\\n',\n          TRANSFORM(\n            SORT_ARRAY(COLLECT_LIST(NAMED_STRUCT(\n              'seq',TFCResultSeq,'adc',ADC_UPDT,'value',TFCValue\n            ))),\n            x -> x.value\n          )\n        ) AS TFCValue,\n        MAX(ADC_UPDT) AS ADC_UPDT,\n        island_id\n      FROM rl_island\n      GROUP BY LIMSNo,LabNo,TFCCode,WkgCode,island_id\n    ),\n    sl1 AS (\n      SELECT *\n      FROM (\n        SELECT\n          CAST(sl.LIMSNo AS INT) AS LIMSNo, sl.LabNo, sl.MRN, sl.NHSNo,\n          sl.AssAuthCode, sl.RequestDT, sl.SampleDT, sl.ReportDate,\n          sl.ReceiptDT, sl.BookedInDT, sl.OrderNo, sl.VisitID,\n          sl.BodySiteCode, sl.CSpecTypeCode, sl.SpecimenCategory,\n          sl.UrgentFlag, sl.SourceCode, sl.ClinicianCode,\n          sl.ADC_UPDT AS sample_adc,\n          ROW_NUMBER() OVER (\n            PARTITION BY sl.LIMSNo,sl.LabNo\n            ORDER BY sl.ADC_UPDT DESC NULLS LAST,\n                     sl.SampleDT DESC NULLS LAST,\n                     sl.ReportDate DESC NULLS LAST,\n                     sl.ReceiptDT DESC NULLS LAST,\n                     sl.BookedInDT DESC NULLS LAST,\n                     sl.OrderNo ASC NULLS LAST,\n                     sl.VisitID ASC NULLS LAST,\n                     sl.MRN ASC NULLS LAST,\n                     sl.NHSNo ASC NULLS LAST\n          ) rn\n        FROM {_qn(SAMPLE_LEVEL)} sl\n        {raw_scope_sample_join}\n      ) WHERE rn=1\n    ),\n    m1 AS (\n      SELECT *\n      FROM (\n        SELECT\n          m.WkgCode,m.TFCCode,m.TFCDesc_Full,m.TFCDesc_Rep,m.TFCDesc_WP,\n          m.ReportingSynonym,m.PMIPDesc,m.Units,m.NLMC_ID,m.SectionCode,\n          m.WorkSectionCode,m.ReportSection,m.ResultType,m.ResultFormat,\n          m.NumValUpper,m.NumValDPs,m.LastUpdateDT,m.ADC_UPDT,\n          ROW_NUMBER() OVER (\n            PARTITION BY m.WkgCode,m.TFCCode\n            ORDER BY m.LastUpdateDT DESC NULLS LAST,\n                     m.ADC_UPDT DESC NULLS LAST\n          ) rn\n        FROM {_qn(MASTER_RESULT)} m\n      ) WHERE rn=1\n    ),\n    raw_person AS (\n      -- Broadcast MRN/NHS resolver maps so NULL and blank aliases do not\n      -- collapse into long-running sort-merge partitions.\n      SELECT /*+ BROADCAST(mr, nr) */\n        rl.*,sl.* EXCEPT (LIMSNo,LabNo,rn),\n        mr.PERSON_ID AS person_id_mrn,\n        nr.PERSON_ID AS person_id_nhs,\n        mr.person_count AS mrn_person_count,\n        nr.person_count AS nhs_person_count,\n        mr.ADC_UPDT AS mrn_alias_adc,\n        nr.ADC_UPDT AS nhs_alias_adc,\n        CASE\n          WHEN mr.PERSON_ID IS NOT NULL AND nr.PERSON_ID IS NOT NULL\n               AND mr.PERSON_ID<>nr.PERSON_ID THEN CAST(NULL AS BIGINT)\n          ELSE COALESCE(mr.PERSON_ID,nr.PERSON_ID)\n        END AS resolved_person_id,\n        CASE\n          WHEN mr.PERSON_ID IS NOT NULL AND nr.PERSON_ID IS NOT NULL\n               AND mr.PERSON_ID<>nr.PERSON_ID THEN 'conflict'\n          WHEN mr.PERSON_ID IS NOT NULL AND nr.PERSON_ID=mr.PERSON_ID THEN 'agreed'\n          WHEN mr.PERSON_ID IS NOT NULL THEN 'mrn_only'\n          WHEN nr.PERSON_ID IS NOT NULL THEN 'nhs_only'\n          WHEN COALESCE(mr.person_count,0)>1 OR COALESCE(nr.person_count,0)>1 THEN 'ambiguous'\n          ELSE 'unresolved'\n        END AS person_match_status\n      FROM rl\n      LEFT JOIN sl1 sl ON sl.LIMSNo <=> rl.LIMSNo AND sl.LabNo <=> rl.LabNo\n      LEFT JOIN mrn_resolver mr ON mr.ALIAS=sl.MRN\n      LEFT JOIN nhs_resolver nr ON nr.ALIAS=sl.NHSNo\n    ),\n    raw_src AS (\n      -- Broadcast canonical person aliases so unresolved NULL person IDs do not\n      -- collapse into one sort-merge reducer during the full pathology rebuild.\n      SELECT /*+ BROADCAST(pm, pn) */\n        'raw' AS source_table,\n        'TFC_LIMS' AS source_system,\n        CONCAT_WS('|','raw',COALESCE(CAST(rp.LIMSNo AS STRING),'∅'),\n                  COALESCE(rp.LabNo,'∅')) AS source_parent_key,\n        CONCAT_WS('|','raw',COALESCE(CAST(rp.LIMSNo AS STRING),'∅'),\n                  COALESCE(rp.LabNo,'∅'),COALESCE(rp.TFCCode,'∅'),\n                  COALESCE(rp.WkgCode,'∅'),CAST(rp.island_id AS STRING))\n          AS source_record_key,\n        COALESCE(\n          rp.TFCResultSeq,\n          XXHASH64(CONCAT_WS('|',COALESCE(CAST(rp.LIMSNo AS STRING),'∅'),\n            COALESCE(rp.LabNo,'∅'),COALESCE(rp.TFCCode,'∅'),\n            COALESCE(rp.WkgCode,'∅'),CAST(rp.island_id AS STRING)))\n        ) AS source_event_id,\n        (rp.TFCResultSeq IS NULL) AS is_synthetic_key,\n        rp.LabNo AS lab_no,\n        rp.LIMSNo,\n        rp.TFCResultSeq AS source_sequence_start,\n        rp.TFCResultSeqEnd AS source_sequence_end,\n        CAST(rp.source_line_count AS BIGINT) AS source_line_count,\n        rp.MonthYear AS source_month_year,\n        rp.MonthYearAuth AS source_month_year_auth,\n        CAST(rp.resolved_person_id AS BIGINT) AS PERSON_ID,\n        CAST(NULL AS BIGINT) AS ENCNTR_ID,\n        CAST(rp.person_id_mrn AS BIGINT) AS person_id_mrn,\n        CAST(rp.person_id_nhs AS BIGINT) AS person_id_nhs,\n        rp.person_match_status,\n        (rp.person_match_status='conflict') AS person_match_conflict,\n        COALESCE(pm.canonical_mrn,rp.MRN) AS MRN,\n        COALESCE(pn.canonical_nhs,rp.NHSNo) AS NHS_Number,\n        CAST(NULL AS INT) AS EVENT_CD,\n        CAST(NULL AS STRING) AS EVENT_CD_DISPLAY,\n        COALESCE(rp.SampleDT,rp.RequestDT,rp.ReceiptDT,rp.BookedInDT,rp.ReportDate)\n          AS measurement_datetime,\n        CASE WHEN rp.SampleDT IS NOT NULL THEN 'SampleDT'\n             WHEN rp.RequestDT IS NOT NULL THEN 'RequestDT'\n             WHEN rp.ReceiptDT IS NOT NULL THEN 'ReceiptDT'\n             WHEN rp.BookedInDT IS NOT NULL THEN 'BookedInDT'\n             WHEN rp.ReportDate IS NOT NULL THEN 'ReportDate'\n        END AS measurement_datetime_source,\n        'TFC' AS code_system,\n        rp.TFCCode AS code,\n        CASE WHEN NULLIF(TRIM(rp.TFCCode),'') IS NOT NULL\n                  AND rp.TFCCode <=> rp.LegTFCCode THEN 'LegTFCCode'\n             WHEN NULLIF(TRIM(rp.TFCCode),'') IS NOT NULL THEN 'TFCCode'\n             ELSE 'missing'\n        END AS code_source,\n        COALESCE(m1.TFCDesc_Full,m1.TFCDesc_Rep,m1.TFCDesc_WP,\n                 m1.ReportingSynonym,m1.PMIPDesc,rp.TFCCode) AS description,\n        CASE WHEN m1.TFCDesc_Full IS NOT NULL THEN 'TFCDesc_Full'\n             WHEN m1.TFCDesc_Rep IS NOT NULL THEN 'TFCDesc_Rep'\n             WHEN m1.TFCDesc_WP IS NOT NULL THEN 'TFCDesc_WP'\n             WHEN m1.ReportingSynonym IS NOT NULL THEN 'ReportingSynonym'\n             WHEN m1.PMIPDesc IS NOT NULL THEN 'PMIPDesc'\n             WHEN rp.TFCCode IS NOT NULL THEN 'TFCCode'\n             ELSE 'missing'\n        END AS description_source,\n        rp.TFCValue AS result_txt,\n        m1.Units AS unit_source_value,\n        CAST(NULL AS STRING) AS range_low_raw,\n        CAST(NULL AS STRING) AS range_high_raw,\n        CAST(NULL AS DOUBLE) AS range_low,\n        CAST(NULL AS DOUBLE) AS range_high,\n        CAST(NULL AS STRING) AS normalcy,\n        rp.WkgCode,\n        m1.NLMC_ID AS nlmc_id,\n        rp.ReportDate,\n        GREATEST(rp.ADC_UPDT,rp.sample_adc,rp.mrn_alias_adc,rp.nhs_alias_adc,\n                 m1.ADC_UPDT,pm.canonical_mrn_adc,pn.canonical_nhs_adc)\n          AS source_adc_updt,\n        CAST(NULL AS STRING) AS reference_nbr,\n        CAST(NULL AS BIGINT) AS clinical_event_id,\n        CAST(NULL AS BIGINT) AS order_id,\n        CAST(NULL AS BIGINT) AS catalog_cd,\n        CAST(NULL AS BIGINT) AS parent_event_id,\n        CAST(NULL AS BIGINT) AS event_reltn_cd,\n        CAST(NULL AS TIMESTAMP) AS valid_from_dt_tm,\n        CAST(NULL AS TIMESTAMP) AS valid_until_dt_tm,\n        CAST(NULL AS TIMESTAMP) AS event_start_dt_tm,\n        CAST(NULL AS TIMESTAMP) AS event_end_dt_tm,\n        CAST(NULL AS TIMESTAMP) AS performed_dt_tm,\n        CAST(NULL AS TIMESTAMP) AS verified_dt_tm,\n        CAST(NULL AS BIGINT) AS record_status_cd,\n        CAST(NULL AS BIGINT) AS result_status_cd,\n        CAST(NULL AS BIGINT) AS authentic_flag,\n        CAST(NULL AS TIMESTAMP) AS clinsig_updt_dt_tm,\n        CAST(NULL AS BIGINT) AS source_updt_cnt,\n        CAST(NULL AS BIGINT) AS contributor_system_cd,\n        CAST(NULL AS BIGINT) AS result_units_cd,\n        CAST(NULL AS BIGINT) AS normalcy_cd,\n        rp.LegWkgCode AS legacy_wkg_code,\n        rp.LegTFCCode AS legacy_tfc_code,\n        rp.RequestDT AS request_dt,\n        rp.SampleDT AS sample_dt,\n        rp.ReceiptDT AS receipt_dt,\n        rp.BookedInDT AS booked_in_dt,\n        rp.OrderNo AS order_no,\n        rp.VisitID AS visit_id,\n        rp.AssAuthCode AS ass_auth_code,\n        rp.BodySiteCode AS body_site_code,\n        rp.CSpecTypeCode AS specimen_type_code,\n        rp.SpecimenCategory AS specimen_category,\n        rp.UrgentFlag AS urgent_flag,\n        rp.SourceCode AS source_code,\n        rp.ClinicianCode AS clinician_code,\n        m1.SectionCode AS master_section_code,\n        m1.WorkSectionCode AS work_section_code,\n        m1.ReportSection AS report_section,\n        m1.ResultType AS master_result_type,\n        m1.ResultFormat AS master_result_format,\n        CAST(m1.NumValUpper AS INT) AS master_num_val_upper,\n        CAST(m1.NumValDPs AS INT) AS master_num_val_dps\n      FROM raw_person rp\n      LEFT JOIN m1 ON m1.WkgCode <=> rp.WkgCode AND m1.TFCCode <=> rp.TFCCode\n      LEFT JOIN pa_person_mrn pm ON pm.PERSON_ID=rp.resolved_person_id\n      LEFT JOIN pa_person_nhs pn ON pn.PERSON_ID=rp.resolved_person_id\n    ),\n    source_union AS (\n      SELECT * FROM linked_src\n      UNION ALL\n      SELECT * FROM raw_src\n    ),\n    combined AS (\n      SELECT s.*,\n        XXHASH64(\n          source_parent_key,source_record_key,PERSON_ID,ENCNTR_ID,MRN,NHS_Number,\n          measurement_datetime,code,description,result_txt,unit_source_value,\n          range_low_raw,range_high_raw,normalcy,WkgCode,nlmc_id,ReportDate,\n          reference_nbr,clinical_event_id,order_id,catalog_cd,valid_until_dt_tm,\n          source_adc_updt,request_dt,sample_dt,receipt_dt,booked_in_dt\n        ) AS source_payload_hash\n      FROM source_union s\n    ),\n    test_map_ranked AS (\n      SELECT *\n      FROM (\n        SELECT tm.*,\n          ROW_NUMBER() OVER (\n            PARTITION BY code_system,code,description\n            ORDER BY CASE confidence_tier\n                       WHEN 'curated' THEN 1 WHEN 'auto_high' THEN 2\n                       WHEN 'auto_low' THEN 3 ELSE 9 END,\n                     mapping_version DESC NULLS LAST,mapped_at DESC NULLS LAST\n          ) rn\n        FROM {_qn(TEST_MAP)} tm\n        WHERE confidence_tier IN {test_tiers}\n          AND measurement_concept_id IS NOT NULL\n      ) WHERE rn=1\n    ),\n    test_code_safe AS (\n      SELECT code_system,code,MIN(measurement_concept_id) AS measurement_concept_id,\n             MIN(concept_name) AS concept_name,'safe_code' AS confidence_tier\n      FROM test_map_ranked\n      GROUP BY code_system,code\n      HAVING COUNT(DISTINCT measurement_concept_id)=1\n    ),\n    test_joined AS (\n      SELECT c.*,\n        COALESCE(tm.measurement_concept_id,nt.measurement_concept_id,\n                 tc.measurement_concept_id) AS measurement_concept_id,\n        COALESCE(tm.concept_name,nt.concept_name,tc.concept_name)\n          AS measurement_concept_name,\n        COALESCE(tm.confidence_tier,nt.confidence_tier,tc.confidence_tier)\n          AS test_confidence_tier,\n        CASE WHEN tm.measurement_concept_id IS NOT NULL THEN 'exact_context'\n             WHEN nt.measurement_concept_id IS NOT NULL AND c.source_table='linked'\n               THEN 'native_event_cd'\n             WHEN nt.measurement_concept_id IS NOT NULL THEN 'native_nlmc'\n             WHEN tc.measurement_concept_id IS NOT NULL THEN 'safe_code'\n             ELSE 'unmapped'\n        END AS test_mapping_match_type\n      FROM combined c\n      LEFT JOIN test_map_ranked tm\n        ON tm.code_system=c.code_system\n       AND tm.code <=> c.code\n       AND tm.description <=> c.description\n      LEFT JOIN {_qn(MP_NATIVE_TEST)} nt\n        ON nt.key_type=CASE WHEN c.source_table='linked' THEN 'EVENT_CD' ELSE 'NLMC_ID' END\n       AND nt.key_value <=> CASE WHEN c.source_table='linked'\n                                THEN CAST(c.EVENT_CD AS STRING) ELSE c.nlmc_id END\n      LEFT JOIN test_code_safe tc\n        ON tc.code_system=c.code_system AND tc.code <=> c.code\n    ),\n    result_derived AS (\n      SELECT t.*,\n        CASE WHEN result_txt RLIKE '{numeric_re}' THEN 1 ELSE 0 END AS rd_result_numeric,\n        CASE WHEN result_txt RLIKE '^\\\\s*<=' THEN 4171754\n             WHEN result_txt RLIKE '^\\\\s*>=' THEN 4171755\n             WHEN result_txt RLIKE '^\\\\s*[<]' THEN 4171756\n             WHEN result_txt RLIKE '^\\\\s*[>]' THEN 4172704\n             WHEN result_txt RLIKE '^\\\\s*≤' THEN 4171754\n             WHEN result_txt RLIKE '^\\\\s*≥' THEN 4171755\n             ELSE NULL\n        END AS rd_operator_concept_id,\n        CASE WHEN result_txt RLIKE '{numeric_re}'\n             THEN TRY_CAST(\n               REGEXP_REPLACE(TRIM(result_txt),'^(?:<=|>=|<|>|≤|≥|=)\\\\s*','')\n               AS DOUBLE\n             )\n        END AS rd_value_as_number,\n        CASE WHEN result_txt IS NOT NULL AND TRIM(result_txt)<>''\n                   AND NOT (result_txt RLIKE '{numeric_re}')\n             THEN LOWER(TRIM(REGEXP_REPLACE(result_txt,'\\\\s+',' ')))\n        END AS rd_result_normalized,\n        CASE WHEN result_txt IS NOT NULL AND TRIM(result_txt)<>''\n                   AND NOT (result_txt RLIKE '{numeric_re}')\n             THEN COALESCE(\n               TRY_TO_TIMESTAMP(TRIM(result_txt),'dd.MM.yyyy'),\n               TRY_TO_TIMESTAMP(TRIM(result_txt),'dd/MM/yyyy'),\n               TRY_TO_TIMESTAMP(TRIM(result_txt),'yyyy-MM-dd'),\n               TRY_TO_TIMESTAMP(TRIM(result_txt),'dd.MM.yy'),\n               TRY_TO_TIMESTAMP(TRIM(result_txt),'dd/MM/yy')\n             )\n        END AS rd_value_as_datetime\n      FROM test_joined t\n    ),\n    result_map_ranked AS (\n      SELECT *\n      FROM (\n        SELECT rm.*,\n          ROW_NUMBER() OVER (\n            PARTITION BY code_system,code,description,result_normalized\n            ORDER BY CASE confidence_tier\n                       WHEN 'curated' THEN 1 WHEN 'auto_high' THEN 2\n                       WHEN 'auto_anchor' THEN 3 WHEN 'auto_value' THEN 4\n                       WHEN 'auto_genpos' THEN 5 WHEN 'auto_low' THEN 6 ELSE 9 END,\n                     mapping_version DESC NULLS LAST,mapped_at DESC NULLS LAST\n          ) rn\n        FROM {_qn(RESULT_MAP)} rm\n        WHERE confidence_tier IN {result_tiers}\n          AND value_as_concept_id IS NOT NULL\n      ) WHERE rn=1\n    ),\n    result_code_safe AS (\n      SELECT code_system,code,result_normalized,\n             MIN(value_as_concept_id) AS value_as_concept_id,\n             MIN(concept_name) AS concept_name,\n             MIN(confidence_tier) AS confidence_tier,\n             (MAX(CAST(is_suspected AS INT))=1) AS is_suspected,\n             MIN(growth_grade) AS growth_grade\n      FROM result_map_ranked\n      GROUP BY code_system,code,result_normalized\n      HAVING COUNT(DISTINCT value_as_concept_id)=1\n    ),\n    result_joined AS (\n      SELECT /*+ BROADCAST(rm, nr, rc) */ d.*,\n        CASE WHEN d.rd_result_numeric=1 THEN CAST(NULL AS BIGINT)\n             ELSE COALESCE(rm.value_as_concept_id,nr.value_as_concept_id,\n                           rc.value_as_concept_id)\n        END AS value_as_concept_id,\n        COALESCE(rm.concept_name,nr.concept_name,rc.concept_name) AS result_concept_name,\n        COALESCE(rm.confidence_tier,nr.confidence_tier,rc.confidence_tier)\n          AS result_confidence_tier,\n        COALESCE(rm.is_suspected,rc.is_suspected) AS result_is_suspected,\n        COALESCE(rm.growth_grade,rc.growth_grade) AS result_growth_grade,\n        CASE WHEN d.rd_result_numeric=1 THEN 'numeric'\n             WHEN rm.value_as_concept_id IS NOT NULL THEN 'exact_context'\n             WHEN nr.value_as_concept_id IS NOT NULL THEN 'native_context'\n             WHEN rc.value_as_concept_id IS NOT NULL THEN 'safe_code_result'\n             ELSE 'unmapped'\n        END AS result_mapping_match_type\n      FROM result_derived d\n      LEFT JOIN result_map_ranked rm\n        ON rm.code_system=d.code_system\n       AND rm.code <=> d.code\n       AND rm.description <=> d.description\n       AND rm.result_normalized <=> d.rd_result_normalized\n       AND d.rd_result_numeric=0\n      LEFT JOIN {_qn(MP_NATIVE_RESULT)} nr\n        ON nr.key_type=CASE WHEN d.source_table='linked' THEN 'EVENT_CD' ELSE 'NLMC_ID' END\n       AND nr.key_value <=> CASE WHEN d.source_table='linked'\n                                THEN CAST(d.EVENT_CD AS STRING) ELSE d.nlmc_id END\n       AND nr.result_normalized <=> d.rd_result_normalized\n       AND d.rd_result_numeric=0\n      LEFT JOIN result_code_safe rc\n        ON rc.code_system=d.code_system\n       AND rc.code <=> d.code\n       AND rc.result_normalized <=> d.rd_result_normalized\n       AND d.rd_result_numeric=0\n    ),\n    unit_exact AS (\n      SELECT unit_source_value,unit_concept_id,ucum_code\n      FROM (\n        SELECT um.*,\n          ROW_NUMBER() OVER (\n            PARTITION BY unit_source_value\n            ORDER BY unit_concept_id DESC NULLS LAST,ucum_code ASC NULLS LAST\n          ) rn\n        FROM {_qn(UNIT_MAP)} um\n        WHERE unit_concept_id IS NOT NULL\n      ) WHERE rn=1\n    ),\n    unit_normalized AS (\n      SELECT LOWER(TRIM(unit_source_value)) AS unit_norm,\n             MIN(unit_concept_id) AS unit_concept_id,\n             MIN(ucum_code) AS ucum_code\n      FROM unit_exact\n      WHERE unit_source_value IS NOT NULL AND TRIM(unit_source_value)<>''\n      GROUP BY LOWER(TRIM(unit_source_value))\n      HAVING COUNT(DISTINCT unit_concept_id)=1\n    ),\n    unit_joined AS (\n      SELECT r.*,\n        COALESCE(ue.unit_concept_id,un.unit_concept_id) AS unit_concept_id,\n        COALESCE(ue.ucum_code,un.ucum_code) AS ucum_code,\n        CASE WHEN ue.unit_concept_id IS NOT NULL THEN 'exact'\n             WHEN un.unit_concept_id IS NOT NULL THEN 'normalized'\n             ELSE 'unmapped'\n        END AS unit_mapping_match_type\n      FROM result_joined r\n      LEFT JOIN unit_exact ue ON ue.unit_source_value <=> r.unit_source_value\n      LEFT JOIN unit_normalized un\n        ON un.unit_norm=LOWER(TRIM(r.unit_source_value))\n       AND ue.unit_concept_id IS NULL\n    ),\n    concept_ids AS (\n      SELECT measurement_concept_id AS concept_id\n      FROM test_map_ranked\n      WHERE measurement_concept_id IS NOT NULL\n      UNION\n      SELECT measurement_concept_id\n      FROM {_qn(MP_NATIVE_TEST)}\n      WHERE measurement_concept_id IS NOT NULL\n      UNION\n      SELECT value_as_concept_id\n      FROM result_map_ranked\n      WHERE value_as_concept_id IS NOT NULL\n      UNION\n      SELECT value_as_concept_id\n      FROM {_qn(MP_NATIVE_RESULT)}\n      WHERE value_as_concept_id IS NOT NULL\n    ),\n    concept_lookup AS (\n      SELECT c.concept_id,c.concept_code,c.standard_concept,c.vocabulary_id\n      FROM {_qn(CONCEPT)} c\n      INNER JOIN concept_ids ids ON ids.concept_id=c.concept_id\n    ),\n    projected AS (\n      SELECT /*+ BROADCAST(mc, rcpt) */\n        u.source_table,\n        CAST(u.source_event_id AS BIGINT) AS source_event_id,\n        u.is_synthetic_key,\n        u.lab_no,\n        CAST(u.PERSON_ID AS BIGINT) AS PERSON_ID,\n        CAST(u.ENCNTR_ID AS BIGINT) AS ENCNTR_ID,\n        u.MRN,u.NHS_Number,u.EVENT_CD,u.EVENT_CD_DISPLAY,\n        u.measurement_datetime,u.code_system,u.code,u.description,\n        CASE WHEN mc.vocabulary_id='SNOMED' THEN mc.concept_code END AS test_snomed_code,\n        CASE WHEN mc.vocabulary_id='LOINC' THEN mc.concept_code END AS test_loinc_code,\n        CAST(u.measurement_concept_id AS BIGINT) AS test_omop_concept_id,\n        mc.standard_concept AS test_omop_standard_concept,\n        mc.vocabulary_id AS test_vocabulary_id,\n        CAST(u.measurement_concept_id AS BIGINT) AS measurement_concept_id,\n        u.measurement_concept_name,u.test_confidence_tier,\n        u.rd_value_as_number AS value_as_number,\n        CAST(u.rd_operator_concept_id AS BIGINT) AS operator_concept_id,\n        CAST(u.value_as_concept_id AS BIGINT) AS value_as_concept_id,\n        u.result_concept_name,u.result_confidence_tier,\n        u.result_is_suspected,u.result_growth_grade,\n        CASE WHEN rcpt.vocabulary_id='SNOMED' THEN rcpt.concept_code END AS result_snomed_code,\n        CASE WHEN rcpt.vocabulary_id='LOINC' THEN rcpt.concept_code END AS result_loinc_code,\n        CAST(u.value_as_concept_id AS BIGINT) AS result_omop_concept_id,\n        rcpt.standard_concept AS result_omop_standard_concept,\n        rcpt.vocabulary_id AS result_vocabulary_id,\n        CASE\n          WHEN u.result_txt IS NULL OR TRIM(u.result_txt)='' THEN 'missing'\n          WHEN u.rd_result_numeric=1 THEN 'numeric'\n          WHEN u.rd_value_as_datetime IS NOT NULL THEN 'datetime'\n          WHEN u.value_as_concept_id IS NOT NULL THEN 'mapped'\n          WHEN u.rd_result_normalized RLIKE '{exclusion_re}' THEN 'excluded'\n          WHEN NOT (u.rd_result_normalized RLIKE '[a-z0-9]') THEN 'sentinel'\n          ELSE 'free_text'\n        END AS result_status,\n        u.result_txt AS value_source_value,\n        u.unit_source_value,u.unit_concept_id,u.ucum_code,\n        u.range_low,u.range_high,u.normalcy,u.WkgCode,u.nlmc_id,u.ReportDate,\n        CURRENT_TIMESTAMP() AS ADC_UPDT,\n        u.source_system,u.source_parent_key,u.source_record_key,\n        u.source_adc_updt,CURRENT_TIMESTAMP() AS loaded_at,\n        CURRENT_TIMESTAMP() AS mapping_updated_at,\n        u.source_payload_hash,\n        XXHASH64(\n          u.measurement_concept_id,u.measurement_concept_name,u.test_confidence_tier,\n          u.value_as_concept_id,u.result_concept_name,u.result_confidence_tier,\n          u.result_is_suspected,u.result_growth_grade,u.unit_concept_id,u.ucum_code,\n          u.rd_value_as_number,u.rd_operator_concept_id\n        ) AS mapping_payload_hash,\n        u.LIMSNo,u.source_sequence_start,u.source_sequence_end,u.source_line_count,\n        u.source_month_year,u.source_month_year_auth,\n        u.person_id_mrn,u.person_id_nhs,u.person_match_status,u.person_match_conflict,\n        u.measurement_datetime_source,u.code_source,u.description_source,\n        u.test_mapping_match_type,u.result_mapping_match_type,u.unit_mapping_match_type,\n        CASE WHEN u.result_txt IS NULL OR TRIM(u.result_txt)='' THEN 'blank'\n             WHEN u.rd_result_numeric=1 THEN 'numeric'\n             WHEN u.rd_value_as_datetime IS NOT NULL THEN 'datetime'\n             ELSE 'text'\n        END AS result_parse_status,\n        u.rd_value_as_datetime AS value_as_datetime,\n        u.range_low_raw,u.range_high_raw,u.reference_nbr,u.clinical_event_id,\n        u.order_id,u.catalog_cd,u.parent_event_id,u.event_reltn_cd,\n        u.valid_from_dt_tm,u.valid_until_dt_tm,u.event_start_dt_tm,u.event_end_dt_tm,\n        u.performed_dt_tm,u.verified_dt_tm,u.record_status_cd,u.result_status_cd,\n        u.authentic_flag,u.clinsig_updt_dt_tm,u.source_updt_cnt,\n        u.contributor_system_cd,u.result_units_cd,u.normalcy_cd,\n        u.legacy_wkg_code,u.legacy_tfc_code,u.request_dt,u.sample_dt,\n        u.receipt_dt,u.booked_in_dt,u.order_no,u.visit_id,u.ass_auth_code,\n        u.body_site_code,u.specimen_type_code,u.specimen_category,u.urgent_flag,\n        u.source_code,u.clinician_code,u.master_section_code,u.work_section_code,\n        u.report_section,u.master_result_type,u.master_result_format,\n        u.master_num_val_upper,u.master_num_val_dps,\n        CONCAT_WS('|',\n          CASE WHEN u.person_match_conflict THEN 'PERSON_ID_CONFLICT' END,\n          CASE WHEN u.PERSON_ID IS NULL THEN 'PERSON_ID_UNRESOLVED' END,\n          CASE WHEN u.measurement_datetime IS NULL THEN 'MEASUREMENT_DATETIME_MISSING' END,\n          CASE WHEN u.measurement_datetime<TIMESTAMP'1900-01-01' THEN 'MEASUREMENT_DATETIME_SENTINEL' END,\n          CASE WHEN u.measurement_datetime>CURRENT_TIMESTAMP()+INTERVAL 1 DAY THEN 'MEASUREMENT_DATETIME_FUTURE' END,\n          CASE WHEN u.code IS NULL OR TRIM(u.code)='' THEN 'CODE_MISSING' END,\n          CASE WHEN u.description IS NULL OR TRIM(u.description)='' THEN 'DESCRIPTION_MISSING' END,\n          CASE WHEN u.result_txt IS NULL OR TRIM(u.result_txt)='' THEN 'RESULT_BLANK' END,\n          CASE WHEN u.is_synthetic_key THEN 'SYNTHETIC_SOURCE_KEY' END,\n          CASE WHEN u.source_table='raw' AND u.LIMSNo IS NULL THEN 'LIMSNO_MISSING' END,\n          CASE WHEN u.source_table='raw' AND u.ReportDate<u.measurement_datetime\n                     AND TO_DATE(u.ReportDate)<TO_DATE(u.measurement_datetime)\n               THEN 'REPORT_BEFORE_SAMPLE_DATE' END\n        ) AS data_quality_flags\n      FROM unit_joined u\n      LEFT JOIN concept_lookup mc ON mc.concept_id=u.measurement_concept_id\n      LEFT JOIN concept_lookup rcpt ON rcpt.concept_id=u.value_as_concept_id\n    )\n    SELECT * FROM projected\n    "

def _stage_table(run_id: str, suffix: str) -> str:
    safe_run = re.sub('[^a-zA-Z0-9_]', '_', run_id)
    return f'{TARGET_SCHEMA}._tmp_map_pathology_v2_{safe_run}_{suffix}'

def _materialize_stage(
    sql_text: str,
    table_name: str,
    repartition_count: int | None = None,
) -> int:
    df = spark.sql(sql_text)
    previous_coalesce = None
    if repartition_count is not None:
        previous_coalesce = spark.conf.get(
            "spark.sql.adaptive.coalescePartitions.enabled",
            "true",
        )
        spark.conf.set(
            "spark.sql.adaptive.coalescePartitions.enabled",
            "false",
        )
        df = df.repartition(
            int(repartition_count),
            F.col("source_record_key"),
        )

    # This full rebuild already has an explicit, key-balanced repartition.
    # Delta optimized writes add a second large shuffle and can leave one
    # pathological reducer after every planned partition has completed.
    write_overrides = {
        "spark.databricks.delta.optimizeWrite.enabled": "false",
        "spark.databricks.delta.autoCompact.enabled": "false",
    }
    previous_write_confs: dict[str, str | None] = {}
    for conf_key, conf_value in write_overrides.items():
        try:
            previous_write_confs[conf_key] = spark.conf.get(conf_key, None)
            spark.conf.set(conf_key, conf_value)
        except Exception as conf_exc:
            print(
                f"[map_pathology_v2] unable to override {conf_key}: "
                f"{str(conf_exc).splitlines()[0]}"
            )

    try:
        (
            df.write.format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .option("delta.autoOptimize.optimizeWrite", "false")
            .option("delta.autoOptimize.autoCompact", "false")
            .saveAsTable(table_name)
        )
    finally:
        if previous_coalesce is not None:
            spark.conf.set(
                "spark.sql.adaptive.coalescePartitions.enabled",
                previous_coalesce,
            )
        for conf_key, previous_value in previous_write_confs.items():
            try:
                if previous_value is None:
                    spark.conf.unset(conf_key)
                else:
                    spark.conf.set(conf_key, previous_value)
            except Exception as restore_exc:
                print(
                    f"[map_pathology_v2] unable to restore {conf_key}: "
                    f"{str(restore_exc).splitlines()[0]}"
                )
    return int(spark.table(table_name).count())

def _validate_stage(table_name: str) -> dict[str, int]:
    stage = spark.table(table_name)
    required = {'source_table', 'source_parent_key', 'source_record_key', 'source_event_id', 'source_payload_hash', 'mapping_payload_hash'}
    missing = required - set(stage.columns)
    if missing:
        raise RuntimeError(f'Stage is missing required columns: {sorted(missing)}')
    null_keys = stage.filter(F.col('source_parent_key').isNull() | F.col('source_record_key').isNull()).limit(1).count()
    if null_keys:
        raise RuntimeError('Stage contains NULL parent or record keys.')
    duplicate = stage.groupBy('source_record_key').count().filter(F.col('count') > 1).orderBy(F.desc('count')).limit(10).collect()
    if duplicate:
        raise RuntimeError('Stage contains duplicate source_record_key values: ' + ', '.join((f"{r['source_record_key']} ({r['count']})" for r in duplicate)))
    invalid_raw_parent = stage.filter(F.col('source_table') == 'raw').filter(~F.col('source_parent_key').startswith('raw|')).limit(1).count()
    if invalid_raw_parent:
        raise RuntimeError('Raw stage contains an invalid source_parent_key.')
    return {'row_count': int(stage.count()), 'parent_count': int(stage.select('source_parent_key').distinct().count())}

def _full_replace(stage_table: str) -> None:
    # Reuse the already balanced stage partitions instead of adding another
    # optimized-write shuffle while copying the full snapshot to the target.
    write_overrides = {
        "spark.databricks.delta.optimizeWrite.enabled": "false",
        "spark.databricks.delta.autoCompact.enabled": "false",
    }
    previous_write_confs: dict[str, str | None] = {}
    for conf_key, conf_value in write_overrides.items():
        try:
            previous_write_confs[conf_key] = spark.conf.get(conf_key, None)
            spark.conf.set(conf_key, conf_value)
        except Exception as conf_exc:
            print(
                f"[map_pathology_v2] unable to override {conf_key}: "
                f"{str(conf_exc).splitlines()[0]}"
            )
    try:
        (
            spark.table(stage_table)
            .write.format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .option("delta.enableChangeDataFeed", "true")
            .option("delta.enableRowTracking", "true")
            .option("delta.autoOptimize.optimizeWrite", "false")
            .option("delta.autoOptimize.autoCompact", "false")
            .saveAsTable(MP_TARGET)
        )
    finally:
        for conf_key, previous_value in previous_write_confs.items():
            try:
                if previous_value is None:
                    spark.conf.unset(conf_key)
                else:
                    spark.conf.set(conf_key, previous_value)
            except Exception as restore_exc:
                print(
                    f"[map_pathology_v2] unable to restore {conf_key}: "
                    f"{str(restore_exc).splitlines()[0]}"
                )
    _apply_table_metadata()

def _merge_and_reconcile(stage_table: str, touched_parents: DataFrame) -> dict[str, int]:
    """
    Merge current rows, then delete obsolete children under touched parents.

    Because the stage contains every child for each touched parent, this safely
    handles mutable result text/descriptions, re-keying, and CDF delete events.
    """
    source = spark.table(stage_table)
    target = spark.table(MP_TARGET)
    source_keys = source.select('source_record_key').dropDuplicates()
    parent_keys = touched_parents.select('source_parent_key').dropDuplicates()
    changed_count = int(source.alias('s').join(target.select('source_record_key', 'source_payload_hash', 'mapping_payload_hash').alias('t'), 'source_record_key', 'left').filter(F.col('t.source_record_key').isNull() | ~F.col('s.source_payload_hash').eqNullSafe(F.col('t.source_payload_hash')) | ~F.col('s.mapping_payload_hash').eqNullSafe(F.col('t.mapping_payload_hash'))).count())
    DeltaTable.forName(spark, MP_TARGET).alias('t').merge(source.alias('s'), 't.source_record_key=s.source_record_key').whenMatchedUpdateAll(condition='NOT (t.source_payload_hash <=> s.source_payload_hash) OR NOT (t.mapping_payload_hash <=> s.mapping_payload_hash)').whenNotMatchedInsertAll().execute()
    stale = spark.table(MP_TARGET).join(parent_keys, 'source_parent_key', 'inner').join(source_keys, 'source_record_key', 'left_anti').select('source_record_key').dropDuplicates()
    stale_count = int(stale.count())
    if stale_count:
        DeltaTable.forName(spark, MP_TARGET).alias('t').merge(stale.alias('s'), 't.source_record_key=s.source_record_key').whenMatchedDelete().execute()
    None
    return {'changed_rows': changed_count, 'stale_rows_deleted': stale_count}

def _drop_table_if_exists(table_name: str) -> None:
    spark.sql(f'DROP TABLE IF EXISTS {_qn(table_name)}')

def discover_new_keys_from_stage(stage_table: str) -> dict[str, int]:
    """
    Queue unmapped keys from the exact rows already materialized for this run.

    This replaces the old source rescan and uses result_parse_status from the
    shared parser, eliminating the permissive [0-9.]+ numeric drift.
    """
    queue_name = globals().get('QUEUE', f'{MAP_SCHEMA}.pathology_embed_queue')
    embeddings_name = globals().get('EMBEDDINGS_TABLE', '3_lookup.embeddings.terms')
    if not _table_exists(queue_name):
        raise RuntimeError(f'Embedding queue does not exist: {queue_name}')
    stage = spark.table(stage_table)
    tests = stage.filter(F.col('measurement_concept_id').isNull() & F.col('code').isNotNull() & F.col('description').isNotNull()).select('code_system', 'code', 'description', F.lit(None).cast('string').alias('result_normalized'), F.concat_ws(' | ', F.lower(F.col('code')), F.lower(F.col('description'))).alias('term'), F.lit('test').alias('kind')).dropDuplicates(['code_system', 'code', 'description', 'kind'])
    results = stage.filter(F.col('value_as_concept_id').isNull() & (F.col('result_parse_status') == 'text') & (F.col('result_status') == 'free_text') & F.col('value_source_value').isNotNull()).withColumn('result_normalized', F.lower(F.trim(F.regexp_replace(F.col('value_source_value'), '\\s+', ' ')))).filter((F.col('result_normalized') != '') & F.col('result_normalized').rlike('[a-z0-9]')).select('code_system', 'code', 'description', 'result_normalized', F.concat_ws(' | ', F.lower(F.coalesce(F.col('description'), F.col('code'))), F.col('result_normalized')).alias('term'), F.lit('result').alias('kind')).dropDuplicates(['code_system', 'code', 'description', 'result_normalized', 'kind'])
    candidates = tests.unionByName(results).withColumn('term_norm', F.lower(F.col('term'))).withColumn('embed_text', F.when(F.col('kind') == 'result', F.col('result_normalized')).otherwise(F.col('term')))
    queue = spark.table(queue_name).select('code_system', 'code', 'description', 'kind', 'term', 'result_normalized')
    anti_condition = (candidates.code_system == queue.code_system) & candidates.code.eqNullSafe(queue.code) & candidates.description.eqNullSafe(queue.description) & (candidates.kind == queue.kind) & candidates.result_normalized.eqNullSafe(queue.result_normalized) & (F.lower(candidates.term) == F.lower(queue.term))
    missing = candidates.join(queue, anti_condition, 'left_anti')
    embedded = spark.table(embeddings_name).select(F.lower(F.col('term')).alias('_embedded_term')).filter(F.col('_embedded_term').isNotNull()).dropDuplicates()
    queued = missing.join(embedded, F.lower(missing.embed_text) == embedded._embedded_term, 'left').withColumn('status', F.when(F.col('_embedded_term').isNotNull(), F.lit('vector_ready')).otherwise(F.lit('pending'))).drop('_embedded_term')
    counts = {(r['kind'], r['status']): int(r['count']) for r in queued.groupBy('kind', 'status').count().collect()}
    queue_columns = set(_column_names(queue_name))
    output_columns = ['code_system', 'code', 'description', 'term', 'kind', 'status', 'term_norm', 'result_normalized', 'embed_text']
    output_columns = [c for c in output_columns if c in queue_columns]
    if queued.limit(1).count():
        queued.select(*output_columns).write.mode('append').saveAsTable(queue_name)
    return {'test_keys': sum((v for (kind, _), v in counts.items() if kind == 'test')), 'result_keys': sum((v for (kind, _), v in counts.items() if kind == 'result')), 'pending': sum((v for (_, status), v in counts.items() if status == 'pending')), 'vector_ready': sum((v for (_, status), v in counts.items() if status == 'vector_ready'))}

def _current_mapping_snapshot() -> DataFrame:
    """Return one deterministic row per consumed test/result/unit map key."""
    return spark.sql(f"\n        WITH tm AS (\n          SELECT *\n          FROM (\n            SELECT t.*,\n              ROW_NUMBER() OVER (\n                PARTITION BY code_system,code,description\n                ORDER BY CASE confidence_tier\n                           WHEN 'curated' THEN 1 WHEN 'auto_high' THEN 2\n                           WHEN 'auto_low' THEN 3 ELSE 9 END,\n                         mapping_version DESC NULLS LAST,mapped_at DESC NULLS LAST\n              ) rn\n            FROM {_qn(TEST_MAP)} t\n            WHERE confidence_tier IN {_tier_sql(TEST_TIERS)}\n          ) WHERE rn=1\n        ),\n        rm AS (\n          SELECT *\n          FROM (\n            SELECT r.*,\n              ROW_NUMBER() OVER (\n                PARTITION BY code_system,code,description,result_normalized\n                ORDER BY CASE confidence_tier\n                           WHEN 'curated' THEN 1 WHEN 'auto_high' THEN 2\n                           WHEN 'auto_anchor' THEN 3 WHEN 'auto_value' THEN 4\n                           WHEN 'auto_genpos' THEN 5 WHEN 'auto_low' THEN 6 ELSE 9 END,\n                         mapping_version DESC NULLS LAST,mapped_at DESC NULLS LAST\n              ) rn\n            FROM {_qn(RESULT_MAP)} r\n            WHERE confidence_tier IN {_tier_sql(RESULT_TIERS)}\n          ) WHERE rn=1\n        ),\n        um AS (\n          SELECT LOWER(TRIM(unit_source_value)) AS unit_norm,\n                 MIN(unit_concept_id) AS unit_concept_id,\n                 MIN(ucum_code) AS ucum_code\n          FROM {_qn(UNIT_MAP)}\n          GROUP BY LOWER(TRIM(unit_source_value))\n          HAVING COUNT(DISTINCT unit_concept_id)<=1\n        )\n        SELECT 'test' AS map_kind,code_system,code,description,\n               CAST(NULL AS STRING) AS result_normalized,\n               CAST(NULL AS STRING) AS unit_norm,\n               CAST(measurement_concept_id AS BIGINT) AS concept_id,\n               XXHASH64(measurement_concept_id,concept_name,confidence_tier,\n                        test_mapping_source,similarity_score) AS payload_hash\n        FROM tm\n        UNION ALL\n        SELECT 'result',code_system,code,description,result_normalized,NULL,\n               CAST(value_as_concept_id AS BIGINT),\n               XXHASH64(value_as_concept_id,concept_name,confidence_tier,\n                        result_mapping_source,similarity_score,is_suspected,growth_grade)\n        FROM rm\n        UNION ALL\n        SELECT 'unit',NULL,NULL,NULL,NULL,unit_norm,\n               CAST(unit_concept_id AS BIGINT),XXHASH64(unit_concept_id,ucum_code)\n        FROM um\n        ").withColumn('key_id', F.sha2(F.to_json(F.struct('map_kind', 'code_system', 'code', 'description', 'result_normalized', 'unit_norm')), 256))

def snapshot_mapping_baseline_v2() -> None:
    snapshot = _current_mapping_snapshot()
    snapshot.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(MP_BASELINE)

def classify_mapping_delta_v2() -> dict:
    current = _current_mapping_snapshot().alias('c')
    if not _table_exists(MP_BASELINE):
        delta = current.withColumn('base_concept_id', F.lit(None).cast('long')).withColumn('delta_type', F.when(F.col('concept_id').isNotNull(), F.lit('additive')).otherwise(F.lit('unchanged')))
    else:
        baseline = spark.table(MP_BASELINE).alias('b')
        delta = current.join(baseline, 'key_id', 'full').select(F.coalesce(F.col('c.map_kind'), F.col('b.map_kind')).alias('map_kind'), F.coalesce(F.col('c.code_system'), F.col('b.code_system')).alias('code_system'), F.coalesce(F.col('c.code'), F.col('b.code')).alias('code'), F.coalesce(F.col('c.description'), F.col('b.description')).alias('description'), F.coalesce(F.col('c.result_normalized'), F.col('b.result_normalized')).alias('result_normalized'), F.coalesce(F.col('c.unit_norm'), F.col('b.unit_norm')).alias('unit_norm'), F.col('c.concept_id').alias('concept_id'), F.col('b.concept_id').alias('base_concept_id'), F.col('c.payload_hash').alias('payload_hash'), F.col('b.payload_hash').alias('base_payload_hash')).withColumn('delta_type', F.when(F.col('base_concept_id').isNull() & F.col('concept_id').isNotNull(), F.lit('additive')).when(~F.col('base_concept_id').eqNullSafe(F.col('concept_id')), F.lit('correction')).when(~F.col('base_payload_hash').eqNullSafe(F.col('payload_hash')), F.lit('metadata')).otherwise(F.lit('unchanged')))
    changed = delta.filter(F.col('delta_type') != 'unchanged')
    counts = {row['delta_type']: int(row['count']) for row in changed.groupBy('delta_type').count().collect()}
    return {'delta_df': changed, 'n_additive_keys': counts.get('additive', 0), 'n_correction_keys': counts.get('correction', 0), 'n_metadata_keys': counts.get('metadata', 0)}

def _parents_affected_by_mapping_delta(delta_df: DataFrame) -> DataFrame:
    target = spark.table(MP_TARGET).withColumn('_result_normalized', F.lower(F.trim(F.regexp_replace(F.col('value_source_value'), '\\s+', ' ')))).withColumn('_unit_norm', F.lower(F.trim(F.col('unit_source_value'))))
    applicable = delta_df.filter(F.col('delta_type').isin('additive', 'metadata'))
    test_delta = applicable.filter(F.col('map_kind') == 'test').alias('d')
    result_delta = applicable.filter(F.col('map_kind') == 'result').alias('d')
    unit_delta = applicable.filter(F.col('map_kind') == 'unit').alias('d')
    test_parents = target.alias('t').join(test_delta, (F.col('t.code_system') == F.col('d.code_system')) & F.col('t.code').eqNullSafe(F.col('d.code')) & F.col('t.description').eqNullSafe(F.col('d.description')), 'inner').select(F.col('t.source_parent_key'))
    result_parents = target.alias('t').join(result_delta, (F.col('t.code_system') == F.col('d.code_system')) & F.col('t.code').eqNullSafe(F.col('d.code')) & F.col('t.description').eqNullSafe(F.col('d.description')) & F.col('t._result_normalized').eqNullSafe(F.col('d.result_normalized')), 'inner').select(F.col('t.source_parent_key'))
    unit_parents = target.alias('t').join(unit_delta, F.col('t._unit_norm').eqNullSafe(F.col('d.unit_norm')), 'inner').select(F.col('t.source_parent_key'))
    return test_parents.unionByName(result_parents).unionByName(unit_parents).dropDuplicates(['source_parent_key'])

def _run_embedding_and_mapping_reconciliation(discovery_stage: str, run_id: str, scratch_tables: list[str]) -> dict[str, int]:
    metrics = {'additive_map_keys': 0, 'correction_map_keys': 0, 'metadata_map_keys': 0, 'mapping_changed_rows': 0, 'mapping_stale_rows_deleted': 0}
    if not ENABLE_EMBED_LOOP:
        return metrics
    required_functions = ('embed_pending_capped', 'remap_keys')
    missing_functions = [name for name in required_functions if name not in globals()]
    if missing_functions:
        print(f'[map_pathology_v2] embed loop skipped; import pathology_embed_increment first. Missing: {missing_functions}')
        return metrics
    discovered = discover_new_keys_from_stage(discovery_stage)
    print(f'[map_pathology_v2] embedding discovery: {discovered}')
    embed_pending_capped()
    remap_keys()
    delta = classify_mapping_delta_v2()
    delta_df = delta['delta_df']
    metrics['additive_map_keys'] = delta['n_additive_keys']
    metrics['correction_map_keys'] = delta['n_correction_keys']
    metrics['metadata_map_keys'] = delta['n_metadata_keys']
    if delta['n_correction_keys'] > 0:
        _set_rebuild_flag(True)
        print(f"[map_pathology_v2] {delta['n_correction_keys']} mapping corrections detected; a true full rebuild is flagged for the next run.")
        None
        return metrics
    affected = _parents_affected_by_mapping_delta(delta_df)
    affected_count = int(affected.count())
    if affected_count:
        print(f"[map_pathology_v2] rebuilding {affected_count:,} parents for {delta['n_additive_keys']:,} additive and {delta['n_metadata_keys']:,} metadata map changes")
        touched = _scope_from_parent_keys(affected)
        map_stage = _stage_table(run_id, 'mapping')
        scratch_tables.append(map_stage)
        _materialize_stage(_mp_build_select(full=False), map_stage)
        _validate_stage(map_stage)
        merge_metrics = _merge_and_reconcile(map_stage, touched)
        metrics['mapping_changed_rows'] = merge_metrics['changed_rows']
        metrics['mapping_stale_rows_deleted'] = merge_metrics['stale_rows_deleted']
    None
    snapshot_mapping_baseline_v2()
    None
    _refresh_native_crosswalks()
    return metrics

def _refresh_map_cutoffs(cutoffs: dict[str, dict]) -> None:
    """Advance map-state cutoffs past mappings created by the inline embed loop."""
    for source_name in ('pathology_test_concept_map', 'pathology_result_concept_map', 'pathology_unit_map'):
        table_name, ts_col = SOURCE_TABLES[source_name]
        cutoffs[source_name] = {'source_name': source_name, 'table_name': table_name, 'end_version': _latest_delta_version(table_name), 'end_adc_updt': _max_timestamp(table_name, ts_col), 'timestamp_column': ts_col}

def create_map_pathology(force_full: bool=False, run_embed_loop: bool=True) -> dict:
    """
    Production entry point and drop-in replacement for Map Pipeline.

    A full rebuild is forced for first deployment, missing v2 state/schema, or
    an outstanding mapping correction. Incremental runs consume per-source CDF
    or timestamp changes and reconcile complete source-parent scopes.
    """
    global ENABLE_EMBED_LOOP
    previous_embed_setting = ENABLE_EMBED_LOOP
    ENABLE_EMBED_LOOP = bool(run_embed_loop)
    _ensure_control_tables()
    run_id = str(uuid.uuid4())
    started_at = datetime.now(timezone.utc).replace(tzinfo=None)
    scratch_tables: list[str] = []
    mode = 'UNKNOWN'
    source_parent_count = 0
    staged_row_count = 0
    inserted_or_updated_rows = 0
    stale_rows_deleted = 0
    additive_map_keys = 0
    correction_map_keys = 0
    previous_full_rebuild_settings: dict[str, str] = {}
    try:
        state = _read_state()
        cutoffs = _capture_cutoffs()
        state_complete = all((name in state for name in SOURCE_TABLES))
        full = bool(force_full) or not _target_is_v2() or (not _table_exists(MP_BASELINE)) or (not state_complete) or _read_rebuild_flag()
        mode = 'FULL_REBUILD' if full else 'INCREMENTAL'
        if full:
            full_rebuild_settings = {
                "spark.sql.shuffle.partitions": "3200",
                "spark.sql.adaptive.coalescePartitions.enabled": "false",
                "spark.sql.broadcastTimeout": "3600",
            }
            full_rebuild_defaults = {
                "spark.sql.shuffle.partitions": "200",
                "spark.sql.adaptive.coalescePartitions.enabled": "true",
                "spark.sql.broadcastTimeout": "300",
            }
            for conf_key, conf_value in full_rebuild_settings.items():
                previous_full_rebuild_settings[conf_key] = spark.conf.get(
                    conf_key,
                    full_rebuild_defaults[conf_key],
                )
                spark.conf.set(conf_key, conf_value)
        print(f'[map_pathology_v2] {mode}; run_id={run_id}')
        _refresh_native_crosswalks()
        source_stage = _stage_table(run_id, 'source')
        scratch_tables.append(source_stage)
        if full:
            staged_row_count = _materialize_stage(_mp_build_select(full=True), source_stage, repartition_count=3200)
            validation = _validate_stage(source_stage)
            source_parent_count = validation['parent_count']
            staged_row_count = validation['row_count']
            _full_replace(source_stage)
            inserted_or_updated_rows = staged_row_count
            snapshot_mapping_baseline_v2()
            _set_rebuild_flag(False)
        else:
            touched_parents, scope_modes = _prepare_incremental_scope(state, cutoffs)
            touched_parents = touched_parents
            source_parent_count = int(touched_parents.count())
            print(f'[map_pathology_v2] touched source parents: {source_parent_count:,}; modes={scope_modes}')
            if source_parent_count:
                staged_row_count = _materialize_stage(_mp_build_select(full=False), source_stage)
                validation = _validate_stage(source_stage)
                staged_row_count = validation['row_count']
                merge_metrics = _merge_and_reconcile(source_stage, touched_parents)
                inserted_or_updated_rows += merge_metrics['changed_rows']
                stale_rows_deleted += merge_metrics['stale_rows_deleted']
            else:
                spark.table(MP_TARGET).limit(0).write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(source_stage)
            None
        mapping_metrics = _run_embedding_and_mapping_reconciliation(source_stage, run_id, scratch_tables)
        additive_map_keys = mapping_metrics['additive_map_keys']
        correction_map_keys = mapping_metrics['correction_map_keys']
        inserted_or_updated_rows += mapping_metrics['mapping_changed_rows']
        stale_rows_deleted += mapping_metrics['mapping_stale_rows_deleted']
        _refresh_map_cutoffs(cutoffs)
        _advance_state(cutoffs)
        completed_at = datetime.now(timezone.utc).replace(tzinfo=None)
        _write_run_log(run_id=run_id, started_at=started_at, completed_at=completed_at, mode=mode, status='SUCCESS', pipeline_version=MP_VERSION, source_parent_count=source_parent_count, staged_row_count=staged_row_count, inserted_or_updated_rows=inserted_or_updated_rows, stale_rows_deleted=stale_rows_deleted, additive_map_keys=additive_map_keys, correction_map_keys=correction_map_keys, message='Completed')
        result = {'run_id': run_id, 'mode': mode, 'source_parent_count': source_parent_count, 'staged_row_count': staged_row_count, 'inserted_or_updated_rows': inserted_or_updated_rows, 'stale_rows_deleted': stale_rows_deleted, 'additive_map_keys': additive_map_keys, 'correction_map_keys': correction_map_keys, 'rebuild_flagged': _read_rebuild_flag()}
        print(f'[map_pathology_v2] complete: {result}')
        return result
    except Exception as exc:
        completed_at = datetime.now(timezone.utc).replace(tzinfo=None)
        message = str(exc)
        _write_run_log(run_id=run_id, started_at=started_at, completed_at=completed_at, mode=mode, status='FAILED', pipeline_version=MP_VERSION, source_parent_count=source_parent_count, staged_row_count=staged_row_count, inserted_or_updated_rows=inserted_or_updated_rows, stale_rows_deleted=stale_rows_deleted, additive_map_keys=additive_map_keys, correction_map_keys=correction_map_keys, message=message[:4000])
        raise
    finally:
        for conf_key, conf_value in previous_full_rebuild_settings.items():
            spark.conf.set(conf_key, conf_value)
        for table_name in scratch_tables:
            try:
                _drop_table_if_exists(table_name)
            except Exception as cleanup_exc:
                print(f'[map_pathology_v2] scratch cleanup warning for {table_name}: {str(cleanup_exc)[:500]}')
        ENABLE_EMBED_LOOP = previous_embed_setting

def validate_map_pathology_v2() -> dict:
    """Read-only deployment checks suitable for a post-run validation cell."""
    if not _target_is_v2():
        raise RuntimeError('Target does not have the v2 schema.')
    target = spark.table(MP_TARGET)
    summary = {row['source_table']: {'rows': int(row['rows']), 'parents': int(row['parents']), 'person_conflicts': int(row['person_conflicts']), 'missing_measurement_datetime': int(row['missing_measurement_datetime']), 'missing_results': int(row['missing_results'])} for row in target.groupBy('source_table').agg(F.count('*').alias('rows'), F.countDistinct('source_parent_key').alias('parents'), F.sum(F.when(F.col('person_match_conflict'), 1).otherwise(0)).alias('person_conflicts'), F.sum(F.when(F.col('measurement_datetime').isNull(), 1).otherwise(0)).alias('missing_measurement_datetime'), F.sum(F.when(F.col('result_status') == 'missing', 1).otherwise(0)).alias('missing_results')).collect()}
    duplicate_keys = target.groupBy('source_record_key').count().filter(F.col('count') > 1).limit(1).count()
    raw_bad_parent = target.filter(F.col('source_table') == 'raw').withColumn('_expected_parent_key', F.concat_ws('|', F.lit('raw'), F.coalesce(F.col('LIMSNo').cast('string'), F.lit('∅')), F.coalesce(F.col('lab_no'), F.lit('∅')))).filter(~F.col('source_parent_key').eqNullSafe(F.col('_expected_parent_key'))).limit(1).count()
    result = {'pipeline_version': MP_VERSION, 'summary': summary, 'duplicate_source_record_keys': int(duplicate_keys), 'raw_parent_key_validation_failures': int(raw_bad_parent), 'rebuild_flagged': _read_rebuild_flag()}
    print(result)
    return result
print('Map Pathology v2 replacement loaded. Call create_map_pathology() explicitly; no production write has run.')

try:
    _targets = ['4_prod.bronze.map_pathology__canonical']
    if not _pipeline_resume_skip_component('map_pathology', _targets):
        create_map_pathology(force_full=_PIPELINE_FULL_REFRESH, run_embed_loop=_PIPELINE_RUN_EMBEDDINGS) if _PIPELINE_RUN_PATHOLOGY else print('[VALIDATION] Pathology candidate skipped')
        _PIPELINE_UPDATED_TARGETS.extend(_targets)
        _pipeline_mark_component_complete('map_pathology', _targets)
        _pipeline_audit(None, 'COMPONENT_END', {'component': 'map_pathology'})
except Exception as exc:
    _pipeline_record_error(None, exc)
    raise
finally:
    if _pipeline_shared_update_table is not None:
        update_table = _pipeline_shared_update_table
    if _pipeline_shared_table_exists is not None:
        table_exists = _pipeline_shared_table_exists
    if _pipeline_shared_get_max_timestamp is not None:
        get_max_timestamp = _pipeline_shared_get_max_timestamp
    if _pipeline_shared_has_cdf_enabled is not None:
        has_cdf_enabled = _pipeline_shared_has_cdf_enabled
    if _pipeline_shared_get_incremental is not None:
        get_incremental_data_with_cdf = _pipeline_shared_get_incremental